In [20]:
# Analysis 1

import os, re
import pandas as pd
import numpy as np

BASE_DIR = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study"
FINAL_DIR = os.path.join(BASE_DIR, "Data", "Final Data")
OUT_DIR  = os.path.join(BASE_DIR, "Data", "Data for Dashboard", "Analysis 1")
os.makedirs(OUT_DIR, exist_ok=True)

HOLDINGS_CSV = os.path.join(FINAL_DIR, "holdings_2025_final.csv")
ETFS_XLSX    = os.path.join(FINAL_DIR, "Selected ESG ETFs.xlsx")
CLASSBIN_CSV = os.path.join(FINAL_DIR, "classification_binary.csv")

OUT_SUMMARY   = os.path.join(OUT_DIR, "context_summary_2025.csv")
OUT_BREAKDOWN = os.path.join(OUT_DIR, "context_breakdown_by_screen.csv")
OUT_EXPLORER  = os.path.join(OUT_DIR, "holdings_explorer_2025.csv")
OUT_SPOTLIGHT = os.path.join(OUT_DIR, "top_holdings_spotlight.csv")

def norm(s):
    return re.sub(r"\s+"," ",str(s)).strip()

def low(s):
    return norm(s).lower()

def to_num(x):
    return pd.to_numeric(x, errors="coerce")

def find_col_like(df, needles):
    for c in df.columns:
        lc = low(c)
        if all(n in lc for n in needles):
            return c
    return None

def build_screen_tags(df):
    order = ["Deforestation","Fossil Fuel","Prison","Tobacco","Weapons","Clean200"]
    def f(r):
        tags=[]
        if r.get("deforestation",0)==1: tags.append("Deforestation")
        if r.get("fossil fuel",0)==1:   tags.append("Fossil Fuel")
        if r.get("prison",0)==1:        tags.append("Prison")
        if r.get("tobacco",0)==1:       tags.append("Tobacco")
        if r.get("weapons",0)==1:       tags.append("Weapons")
        if r.get("clean200",0)==1:      tags.append("Clean200")
        tags_sorted=[t for t in order if t in tags]
        return " | ".join(tags_sorted)
    return df.apply(f, axis=1)

hold = pd.read_csv(HOLDINGS_CSV, dtype=str)
hold["ETF_Ticker"] = hold["ETF_Ticker"].astype(str).str.upper().str.strip()
hold["company_ticker"] = hold["company_ticker"].astype(str).str.upper().str.strip()
hold["name_normalized"] = hold["name_normalized"].astype(str).apply(norm)
hold["Weight (%)"] = to_num(hold["Weight (%)"])
hold = hold.rename(columns={"Weight (%)":"weight_pct_in_etf","Sector":"sector","Location":"region"})

etf_raw = pd.read_excel(ETFS_XLSX, sheet_name=0, dtype=str)
ticker_col = find_col_like(etf_raw, ["ticker"]) or "Ticker"
name_col   = find_col_like(etf_raw, ["name"]) or "Name"
aum_col    = next(c for c in etf_raw.columns if ("net assets" in low(c) and "usd" in low(c)))
asof_col   = find_col_like(etf_raw, ["net","assets","as","of"]) or "Net Assets as of"
etf = etf_raw[[ticker_col,name_col,aum_col,asof_col]].copy()
etf.columns = ["Ticker","Name","Net Assets (USD)","Net Assets as of"]
etf["Ticker"] = etf["Ticker"].astype(str).str.upper().str.strip()
etf["Name"] = etf["Name"].astype(str).apply(norm)
etf["Net Assets (USD)"] = to_num(etf["Net Assets (USD)"].astype(str).str.replace(",","").str.replace("$","").str.strip())
etf["Net Assets as of"] = pd.to_datetime(etf["Net Assets as of"], errors="coerce")
as_of_date = etf["Net Assets as of"].dropna().dt.date.mode().iloc[0].isoformat() if etf["Net Assets as of"].notna().any() else "2025-01-01"
etf = etf.rename(columns={"Ticker":"etf_ticker","Name":"etf_name","Net Assets (USD)":"aum_usd"})

cls = pd.read_csv(CLASSBIN_CSV, dtype=str)
cls["ETF Ticker"] = cls["ETF Ticker"].astype(str).str.upper().str.strip()
cls["company_ticker"] = cls["company_ticker"].astype(str).str.upper().str.strip()
cls["name_normalized"] = cls["name_normalized"].astype(str).apply(norm)
for c in ["clean200","deforestation","fossil fuel","prison","tobacco","weapons"]:
    if c not in cls.columns: cls[c]=0
for c in ["clean200","deforestation","fossil fuel","prison","tobacco","weapons"]:
    cls[c]=to_num(cls[c]).fillna(0).astype(int)
cls_agg = cls.groupby(["ETF Ticker","company_ticker","name_normalized"], as_index=False)[["clean200","deforestation","fossil fuel","prison","tobacco","weapons"]].max()
cls_agg = cls_agg.rename(columns={"ETF Ticker":"ETF_Ticker"})

base = hold.merge(etf[["etf_ticker","etf_name","aum_usd"]], left_on="ETF_Ticker", right_on="etf_ticker", how="inner")
base = base.merge(cls_agg, on=["ETF_Ticker","company_ticker","name_normalized"], how="left")
for c in ["clean200","deforestation","fossil fuel","prison","tobacco","weapons"]:
    base[c] = base[c].fillna(0).astype(int)
base["classification"] = np.where((base[["deforestation","fossil fuel","prison","tobacco","weapons"]].sum(axis=1)>0),"Controversial",np.where(base["clean200"]==1,"Clean","Other"))
base["screen_categories"] = build_screen_tags(base)
base["weight_usd_in_agg"] = to_num(base["aum_usd"]).fillna(0) * to_num(base["weight_pct_in_etf"]).fillna(0) / 100.0
total_aum = base[["etf_ticker","aum_usd"]].drop_duplicates()["aum_usd"].sum()
num_etfs = base["etf_ticker"].nunique()
total_rows = len(base)
uniq_by_name = base["name_normalized"].nunique()

summary = base.groupby("classification", as_index=False)["weight_usd_in_agg"].sum().rename(columns={"weight_usd_in_agg":"exposure_usd"})
summary["share_of_total_aum_pct"] = np.where(total_aum>0, 100.0*summary["exposure_usd"]/total_aum, 0.0)
summary["total_aum_usd"] = total_aum
summary["num_etfs_in_scope"] = num_etfs
summary["total_holdings_rows"] = total_rows
summary["unique_holdings_by_name_global"] = uniq_by_name
summary["as_of_date"] = as_of_date
summary = summary.sort_values("classification")
summary.to_csv(OUT_SUMMARY, index=False)

def explode_overlapping(df):
    rows=[]
    for _,r in df.iterrows():
        sc = str(r.get("screen_categories","")).strip()
        if not sc:
            continue
        for t in [t.strip() for t in sc.split("|") if t.strip()!=""]:
            if t=="Clean200":
                if str(r.get("classification",""))=="Clean":
                    rows.append((t, "Clean", r["weight_usd_in_agg"]))
            else:
                rows.append((t, "Controversial", r["weight_usd_in_agg"]))
    if not rows:
        return pd.DataFrame(columns=["screen_category","classification","exposure_usd","share_of_total_aum_pct","as_of_date"])
    out = pd.DataFrame(rows, columns=["screen_category","classification","exposure_usd"])
    agg = out.groupby(["screen_category","classification"], as_index=False)["exposure_usd"].sum()
    total_aum = df[["etf_ticker","aum_usd"]].drop_duplicates()["aum_usd"].sum()
    agg["share_of_total_aum_pct"] = np.where(total_aum>0, 100.0*agg["exposure_usd"]/total_aum, 0.0)
    agg["as_of_date"] = df.get("as_of_date", pd.Series(["2025-01-01"])).iloc[0] if "as_of_date" in df.columns else "2025-01-01"
    return agg.sort_values(["classification","exposure_usd"], ascending=[True,False])

breakdown = explode_overlapping(base)
breakdown.to_csv(OUT_BREAKDOWN, index=False)

explorer = base.rename(columns={
    "ETF_Ticker":"etf_ticker",
    "name_normalized":"holding_name",
    "company_ticker":"ticker",
    "sector":"sector",
    "region":"region"
})[["etf_ticker","etf_name","holding_name","ticker","sector","region","classification","screen_categories","weight_pct_in_etf","aum_usd","weight_usd_in_agg"]].copy()
explorer["as_of_date"] = as_of_date
explorer.to_csv(OUT_EXPLORER, index=False)

agg_hold = base.groupby(["name_normalized","company_ticker"], as_index=False).agg(
    exposure_usd=("weight_usd_in_agg","sum"),
    num_etfs=("etf_ticker","nunique"),
    screen_def=("screen_categories",lambda x: " | ".join(sorted(set(" | ".join([s for s in x if isinstance(s,str)]).split(" | "))) ) if len(x)>0 else "")
)
agg_flags = base.groupby(["name_normalized","company_ticker"], as_index=False)[["clean200","deforestation","fossil fuel","prison","tobacco","weapons"]].max()
spot = agg_hold.merge(agg_flags, on=["name_normalized","company_ticker"], how="left")
spot["classification"] = np.where((spot[["deforestation","fossil fuel","prison","tobacco","weapons"]].sum(axis=1)>0),"Controversial",np.where(spot["clean200"]==1,"Clean","Other"))
spot["screen_categories"] = spot["screen_def"].replace("",np.nan).fillna(spot.apply(lambda r: ("Clean200" if r["clean200"]==1 else ""), axis=1)).str.strip(" |")
spot = spot.drop(columns=["screen_def"])
spot["share_of_total_aum_pct"] = np.where(total_aum>0, 100.0*spot["exposure_usd"]/total_aum, 0.0)
spot = spot.rename(columns={"name_normalized":"holding_name","company_ticker":"ticker"})
spot = spot[spot["classification"].isin(["Controversial","Clean"])].copy()
spot["cohort"] = spot["classification"]
spot = spot.sort_values(["cohort","exposure_usd"], ascending=[True,False])
spot["rank_within_cohort"] = spot.groupby("cohort")["exposure_usd"].rank(method="first", ascending=False).astype(int)
spot_top = spot[spot["rank_within_cohort"]<=10].copy()
spot_top = spot_top[["cohort","rank_within_cohort","holding_name","ticker","exposure_usd","share_of_total_aum_pct","num_etfs","screen_categories"]]
spot_top["as_of_date"] = as_of_date
spot_top = spot_top.sort_values(["cohort","rank_within_cohort"])
spot_top.to_csv(OUT_SPOTLIGHT, index=False)


In [35]:
# Analysis 2

import os, re
import pandas as pd
import numpy as np

BASE_FINAL="/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data"
SOI_DIR=os.path.join(BASE_FINAL,"Combined Past Holdings")
OUT_DIR="/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 2"
AUM_XLSX=os.path.join(BASE_FINAL,"All Net Assets .xlsx")
CLASS_CSV=os.path.join(BASE_FINAL,"classification_binary.csv")
HOLD_2025=os.path.join(BASE_FINAL,"holdings_2025_final.csv")
ALL_HOLD=os.path.join(BASE_FINAL,"all_holdings_2017_2025.csv")
os.makedirs(OUT_DIR,exist_ok=True)
OUT_EXP_BY_FY=os.path.join(OUT_DIR,"exposures_by_fund_year.csv")
OUT_AGG_TRENDS=os.path.join(OUT_DIR,"aggregate_exposure_trends.csv")
OUT_YEAR_CMP=os.path.join(OUT_DIR,"year_compare_summary.csv")
OUT_DISP=os.path.join(OUT_DIR,"exposure_dispersion_stats.csv")
OUT_SCREENS=os.path.join(OUT_DIR,"aggregate_screen_trends.csv")
OUT_MOVERS=os.path.join(OUT_DIR,"movers_by_yearpair.csv")
TICKER_MAP={"MPCT":"SDG"}

def canon_ticker(t):
    if pd.isna(t): return t
    s=str(t).strip().upper().replace("/",".").replace("-",".").replace(" ",".")
    s=re.sub(r"\.+",".",s)
    return TICKER_MAP.get(s,s)

_CORP_SUFFIX=r"\b(inc|inc\.|corp|corporation|plc|ltd|llc|co|co\.|company|sa|nv|ag|spa|s\.p\.a|holdings|group|limited)\b"
def canon_name(s):
    if pd.isna(s): return s
    s=str(s).lower().replace("&"," and ")
    s=re.sub(r"[^a-z0-9 ]+"," ",s)
    s=re.sub(_CORP_SUFFIX,"",s)
    s=re.sub(r"\bclass\s+[abc]\b","",s)
    s=re.sub(r"\s+"," ",s).strip()
    return s

def parse_money_series(s):
    if s.dtype.kind in "biufc": return pd.to_numeric(s,errors="coerce")
    ss=s.astype(str)
    ss=ss.str.replace(r"\(([\d\.,]+)\)",r"-\1",regex=True)
    ss=ss.str.replace(r"[^\d\.\-]","",regex=True)
    return pd.to_numeric(ss,errors="coerce")

def norm_cols(df):
    m={}
    for c in df.columns:
        k=c.strip().lower()
        k=re.sub(r"\s+","_",k)
        k=k.replace("%","pct").replace("normalised","normalized")
        m[c]=k
    df=df.rename(columns=m)
    r={}
    for c in df.columns:
        nc=c
        if nc=="etf": nc="etf_ticker"
        if nc=="ticker": nc="company_ticker"
        if nc in {"name","holding_name"}: nc="name_normalized"
        r[c]=nc
    return df.rename(columns=r)

def coalesce_dupe_columns(df):
    if not df.columns.duplicated().any(): return df
    cols=[]
    for name in df.columns.unique():
        blk=df.loc[:,df.columns==name]
        cols.append(blk.bfill(axis=1).iloc[:,0] if blk.shape[1]>1 else blk.iloc[:,0])
    out=pd.concat(cols,axis=1)
    out.columns=df.columns.unique()
    return out

def find_col(cols,want):
    wl=want.lower()
    for c in cols:
        if wl in c.strip().lower(): return c
    raise KeyError(want)

def boolish_to_int(x):
    if pd.isna(x): return 0
    s=str(x).strip().lower()
    if s in {"1","true","y","yes"}: return 1
    if s in {"0","false","n","no",""}: return 0
    try: return 1 if float(s)!=0 else 0
    except: return 0

def load_classification_etf_aware(path):
    c=pd.read_csv(path,dtype=str)
    c=coalesce_dupe_columns(norm_cols(c))
    for col in ["clean200","deforestation","fossil_fuel","prison","tobacco","weapons"]:
        if col not in c.columns: c[col]="0"
        c[col]=c[col].apply(boolish_to_int).astype(int)
    etf_col=next(k for k in ["etf_ticker","etf","etf ticker"] if k in c.columns)
    tick_col=next(k for k in ["company_ticker","ticker","symbol"] if k in c.columns)
    name_col=next(k for k in ["name_normalized","holding_name","name"] if k in c.columns)
    c["etf_ticker"]=c[etf_col].astype(str).str.upper().str.strip().map(canon_ticker)
    c["company_ticker"]=c[tick_col].astype(str).str.upper().str.strip()
    c["name_normalized"]=c[name_col].astype(str).str.strip()
    c["ticker_key"]=c["company_ticker"].map(canon_ticker)
    c["name_key"]=c["name_normalized"].map(canon_name)
    grp=(c.groupby(["etf_ticker","ticker_key","name_key"],as_index=False)
           .agg({"deforestation":"max","fossil_fuel":"max","prison":"max","tobacco":"max","weapons":"max","clean200":"max"}))
    by_t=(grp.groupby(["etf_ticker","ticker_key"],as_index=False)
           .agg({"deforestation":"max","fossil_fuel":"max","prison":"max","tobacco":"max","weapons":"max","clean200":"max"}))
    by_n=(grp.groupby(["etf_ticker","name_key"],as_index=False)
           .agg({"deforestation":"max","fossil_fuel":"max","prison":"max","tobacco":"max","weapons":"max","clean200":"max"}))
    return grp,by_t,by_n

def load_aum_all_years(path):
    aum_raw=pd.read_excel(path,dtype=str)
    aum_raw.columns=[c.strip() for c in aum_raw.columns]
    aum_etf_col=find_col(aum_raw.columns,"ticker") if any("ticker" in c.lower() for c in aum_raw.columns) else find_col(aum_raw.columns,"etf")
    aum_year_col=find_col(aum_raw.columns,"year")
    aum_val_col=find_col(aum_raw.columns,"net assets")
    aum=aum_raw.rename(columns={aum_etf_col:"etf_ticker",aum_year_col:"year",aum_val_col:"aum_usd"})
    aum["etf_ticker"]=aum["etf_ticker"].astype(str).str.upper().str.strip().map(canon_ticker)
    aum["year"]=pd.to_numeric(aum["year"],errors="coerce").astype("Int64")
    aum["aum_usd"]=pd.to_numeric(aum["aum_usd"].astype(str).str.replace(",","").str.replace("$",""),errors="coerce")
    return aum.dropna(subset=["etf_ticker","year","aum_usd"])

def build_exposures_by_fund_year():
    clf_main,_,_=load_classification_etf_aware(CLASS_CSV)
    frames=[]
    for y in range(2017,2025):
        p=os.path.join(SOI_DIR,f"soi_{y}_final.csv")
        if os.path.exists(p):
            df=pd.read_csv(p,dtype=str)
            df=coalesce_dupe_columns(norm_cols(df))
            if "value" not in df.columns: continue
            df["mv_usd"]=parse_money_series(df["value"])
            df["year"]=y
            df["etf_ticker"]=df["etf_ticker"].map(canon_ticker)
            df["ticker_key"]=df["company_ticker"].map(canon_ticker)
            df["name_key"]=df["name_normalized"].map(canon_name)
            df=df.dropna(subset=["mv_usd"])
            frames.append(df[["etf_ticker","year","ticker_key","name_key","mv_usd"]])
    d25=pd.read_csv(HOLD_2025,dtype=str)
    d25=coalesce_dupe_columns(norm_cols(d25))
    mv_col="market_value" if "market_value" in d25.columns else ("market_value_usd" if "market_value_usd" in d25.columns else None)
    if mv_col is not None:
        d25["mv_usd"]=parse_money_series(d25[mv_col])
        d25["year"]=2025
        d25["etf_ticker"]=d25["etf_ticker"].map(canon_ticker)
        d25["ticker_key"]=d25["company_ticker"].map(canon_ticker)
        d25["name_key"]=d25["name_normalized"].map(canon_name)
        d25=d25.dropna(subset=["mv_usd"])
        frames.append(d25[["etf_ticker","year","ticker_key","name_key","mv_usd"]])
    u=pd.concat(frames,ignore_index=True)
    u=u[u["mv_usd"]>0]
    m=u.merge(clf_main,on=["etf_ticker","ticker_key","name_key"],how="left")
    for c in ["deforestation","fossil_fuel","prison","tobacco","weapons","clean200"]:
        m[c]=pd.to_numeric(m[c],errors="coerce").fillna(0).astype(int)
    is_contro=(m[["deforestation","fossil_fuel","prison","tobacco","weapons"]].sum(axis=1)>0)
    m["classification"]=np.where(is_contro,"Controversial",np.where(m["clean200"]==1,"Clean","Other"))
    aum=load_aum_all_years(AUM_XLSX).rename(columns={"aum_usd":"market_total_value_usd"})
    m=m.merge(aum,on=["etf_ticker","year"],how="left").dropna(subset=["market_total_value_usd"])
    m["weight_pct"]=m["mv_usd"]/m["market_total_value_usd"]*100.0
    g=m.groupby(["etf_ticker","year","classification"],as_index=False)["weight_pct"].sum()
    p=g.pivot(index=["etf_ticker","year"],columns="classification",values="weight_pct").fillna(0.0).reset_index()
    for col in ["Clean","Controversial","Other"]:
        if col not in p.columns: p[col]=0.0
    out=p.merge(aum,on=["etf_ticker","year"],how="left").rename(columns={"etf_ticker":"ETF ticker","Clean":"pct_clean","Controversial":"pct_controversial","Other":"pct_other"})
    out=out[["ETF ticker","year","pct_controversial","pct_other","pct_clean","market_total_value_usd"]]
    out.to_csv(OUT_EXP_BY_FY,index=False)

def build_aggregates():
    df=pd.read_csv(OUT_EXP_BY_FY)
    year_col="year";fund_col="ETF ticker";pc="pct_clean";px="pct_controversial";po="pct_other"
    for c in [year_col,pc,px,po]: df[c]=pd.to_numeric(df[c],errors="coerce")
    ew=df.groupby(year_col,as_index=False)[[pc,px,po]].mean()
    aum=load_aum_all_years(AUM_XLSX).rename(columns={"etf_ticker":"ETF ticker","aum_usd":"aum_usd"})
    df2=df.merge(aum,on=[fund_col,year_col],how="left")
    def wavg(g):
        w=g["aum_usd"].to_numpy();w=np.where(np.isfinite(w)&(w>0),w,0);s=w.sum()
        if s<=0: return pd.Series({pc:np.nan,px:np.nan,po:np.nan,"aum_total_usd":np.nan})
        return pd.Series({pc:np.average(g[pc],weights=w),px:np.average(g[px],weights=w),po:np.average(g[po],weights=w),"aum_total_usd":s})
    aum_true=df2.groupby(year_col,group_keys=False).apply(wavg).reset_index()
    agg=pd.concat([
        pd.DataFrame({"year":ew[year_col],"weighting_mode":"EW","pct_clean":ew[pc],"pct_controversial":ew[px],"pct_other":ew[po],"aum_total_usd":np.nan}),
        pd.DataFrame({"year":aum_true[year_col],"weighting_mode":"AUM_TRUE","pct_clean":aum_true[pc],"pct_controversial":aum_true[px],"pct_other":aum_true[po],"aum_total_usd":aum_true["aum_total_usd"]})
    ],ignore_index=True)
    yc=agg.melt(id_vars=["year","weighting_mode","aum_total_usd"],value_vars=["pct_clean","pct_controversial","pct_other"],var_name="classification",value_name="exposure_pct")
    yc["classification"]=yc["classification"].map({"pct_clean":"Clean","pct_controversial":"Controversial","pct_other":"Other"})
    agg.to_csv(OUT_AGG_TRENDS,index=False)
    yc[["year","weighting_mode","classification","exposure_pct"]].to_csv(OUT_YEAR_CMP,index=False)

def build_dispersion():
    exp=pd.read_csv(OUT_EXP_BY_FY)
    e_year="year";e_pc="pct_clean";e_px="pct_controversial";e_po="pct_other"
    exp[e_year]=pd.to_numeric(exp[e_year],errors="coerce").astype("Int64")
    for c in [e_pc,e_px,e_po]: exp[c]=pd.to_numeric(exp[c],errors="coerce")
    disp=[]
    for tgt,col in [("clean",e_pc),("controversial",e_px),("other",e_po)]:
        g=exp.groupby(e_year,dropna=True)[col].agg(p10=lambda x:np.nanpercentile(x,10),p50="median",p90=lambda x:np.nanpercentile(x,90),std="std").reset_index()
        g.columns=["year","p10","p50","p90","std"];g["target"]=tgt;disp.append(g[["year","target","p10","p50","p90","std"]])
    pd.concat(disp,ignore_index=True).to_csv(OUT_DISP,index=False)

def build_screens_and_movers():
    hold=pd.read_csv(ALL_HOLD,dtype=str)
    hold.columns=hold.columns.str.strip();hold=hold.loc[:,~hold.columns.duplicated()]
    h_fund="ETF_TICKER";h_date="date";h_name="Normalised name";h_ticker="Company_ticker";h_wgt="weight(%)"
    hold["_year"]=pd.to_datetime(hold[h_date],errors="coerce").dt.year.astype("Int64")
    hold[h_wgt]=pd.to_numeric(hold[h_wgt],errors="coerce").fillna(0.0)
    hold[h_fund]=hold[h_fund].astype(str).str.upper().str.strip()
    hold[h_ticker]=hold[h_ticker].astype(str).str.upper().str.strip()
    hold[h_name]=hold[h_name].astype(str).str.strip()
    hold["_ticker_key"]=hold[h_ticker].map(canon_ticker)
    hold["_name_key"]=hold[h_name].map(canon_name)
    hold["_fund_key"]=hold[h_fund].map(canon_ticker)

    aum=load_aum_all_years(AUM_XLSX).rename(columns={"etf_ticker":h_fund,"year":"_year","aum_usd":"__aum_fy"})
    fy_aum=aum.copy();fy_aum=fy_aum.loc[:,~fy_aum.columns.duplicated()]
    aum_by_year=fy_aum.groupby("_year",as_index=False)["__aum_fy"].sum().rename(columns={"__aum_fy":"__aum_total"})

    clf_grp, _, _ = load_classification_etf_aware(CLASS_CSV)
    grp = clf_grp.rename(columns={"etf_ticker":"_fund_key","ticker_key":"_ticker_key","name_key":"_name_key"})

    h_core=(hold[[h_fund,"_year",h_ticker,h_name,h_wgt,"_fund_key","_ticker_key","_name_key"]]
            .merge(fy_aum,on=[h_fund,"_year"],how="left")
            .merge(aum_by_year,on="_year",how="left")
            .merge(grp,on=["_fund_key","_ticker_key","_name_key"],how="left"))

    for c in ["deforestation","fossil_fuel","prison","tobacco","weapons","clean200"]:
        h_core[c]=pd.to_numeric(h_core[c],errors="coerce").fillna(0).astype(int)

    contro_flags=["deforestation","fossil_fuel","prison","tobacco","weapons"]
    h_core["clean200_exclusive"]=np.where((h_core["clean200"]==1)&(h_core[contro_flags].sum(axis=1)==0),1,0)

    screen_long=[]
    for cat,raw in [
        ("Deforestation","deforestation"),
        ("Fossil Fuel","fossil_fuel"),
        ("Prisons","prison"),
        ("Tobacco","tobacco"),
        ("Weapons","weapons"),
        ("Clean200","clean200_exclusive"),
    ]:
        m=pd.to_numeric(h_core[raw],errors="coerce").fillna(0)
        msk=m.gt(0)&h_core["__aum_fy"].notna()
        if msk.any():
            tmp=h_core.loc[msk,["_year",h_wgt,"__aum_fy"]].copy()
            tmp["screen_category"]=cat
            screen_long.append(tmp)

    if screen_long:
        scr=pd.concat(screen_long,ignore_index=True)
        scr["w_aum"]=scr[h_wgt]*scr["__aum_fy"]/100.0
        scr_agg=scr.groupby(["_year","screen_category"],as_index=False)["w_aum"].sum()
        scr_agg=scr_agg.merge(aum_by_year,on="_year",how="left")
        scr_agg["exposure_pct"]=np.where(scr_agg["__aum_total"]>0,(scr_agg["w_aum"]/scr_agg["__aum_total"])*100.0,np.nan)
        scr_agg["classification"]=np.where(scr_agg["screen_category"]=="Clean200","Clean","Controversial")
        scr_agg["weighting_mode"]="AUM_TRUE"
        scr_agg=scr_agg.rename(columns={"_year":"year"})[["year","screen_category","classification","exposure_pct","weighting_mode"]]
    else:
        scr_agg=pd.DataFrame(columns=["year","screen_category","classification","exposure_pct","weighting_mode"])

    def per_fund_exposure(df,flag_col):
        sub=df[df[flag_col]==1].copy()
        if sub.empty:
            return pd.DataFrame(columns=["_year",h_fund,"exposure_pct"])
        exp_fy=sub.groupby(["_year",h_fund],as_index=False)[h_wgt].sum()
        exp_fy=exp_fy.rename(columns={h_wgt:"exposure_pct"})
        return exp_fy

    ew_frames=[]
    for cat,flag in [
        ("Deforestation","deforestation"),
        ("Fossil Fuel","fossil_fuel"),
        ("Prisons","prison"),
        ("Tobacco","tobacco"),
        ("Weapons","weapons"),
        ("Clean200","clean200_exclusive"),
    ]:
        pf=per_fund_exposure(h_core,flag)
        if not pf.empty:
            yr_avg=pf.groupby("_year",as_index=False)["exposure_pct"].mean()
            yr_avg["screen_category"]=cat
            yr_avg["classification"]="Clean" if cat=="Clean200" else "Controversial"
            yr_avg["weighting_mode"]="EW"
            yr_avg=yr_avg.rename(columns={"_year":"year"})
            ew_frames.append(yr_avg[["year","screen_category","classification","exposure_pct","weighting_mode"]])

    ew_agg=pd.concat(ew_frames,ignore_index=True) if ew_frames else pd.DataFrame(columns=["year","screen_category","classification","exposure_pct","weighting_mode"])
    screens_out=pd.concat([scr_agg,ew_agg],ignore_index=True)
    screens_out.to_csv(OUT_SCREENS,index=False)

    yrs=sorted(h_core["_year"].dropna().astype(int).unique().tolist())

    def contrib_by_year(y):
        sub=h_core[h_core["_year"]==y].dropna(subset=["__aum_fy"])
        if sub.empty:
            return pd.DataFrame(columns=["year","ticker","holding_name","classification","contrib_pct_agg","appear_in_n_funds"])
        sub=sub.copy()
        sub["w_aum"]=sub[h_wgt]*sub["__aum_fy"]/100.0
        g=(sub.groupby([h_ticker,h_name],as_index=False)
              .agg(contrib=("w_aum","sum"),
                   nfunds=(h_fund,"nunique"),
                   defo=("deforestation","max"),
                   foss=("fossil_fuel","max"),
                   pris=("prison","max"),
                   toba=("tobacco","max"),
                   weap=("weapons","max"),
                   cleanx=("clean200_exclusive","max")))
        is_contro=(g[["defo","foss","pris","toba","weap"]].sum(axis=1)>0)
        g["classification"]=np.where(is_contro,"Controversial",np.where(g["cleanx"]==1,"Clean","Other"))
        denom=float(aum_by_year.loc[aum_by_year["_year"]==y,"__aum_total"].iloc[0]) if y in aum_by_year["_year"].values else np.nan
        g["contrib_pct_agg"]=g["contrib"]/denom
        g=g.rename(columns={h_ticker:"ticker",h_name:"holding_name"})
        g["year"]=y;g["appear_in_n_funds"]=g["nfunds"]
        return g[["year","ticker","holding_name","classification","contrib_pct_agg","appear_in_n_funds"]]

    pairs=[]
    if yrs:
        by_year=pd.concat([contrib_by_year(y) for y in yrs],ignore_index=True)
        ref_year=2025
        for a in [y for y in yrs if y<ref_year]:
            xa=(by_year[by_year["year"]==a]
                 [["ticker","holding_name","classification","contrib_pct_agg","appear_in_n_funds"]]
                 .rename(columns={"classification":"classification_a",
                                  "contrib_pct_agg":"c_a",
                                  "appear_in_n_funds":"appear_in_n_funds_a"}))
            xb=(by_year[by_year["year"]==ref_year]
                 [["ticker","holding_name","classification","contrib_pct_agg","appear_in_n_funds"]]
                 .rename(columns={"classification":"classification_b",
                                  "contrib_pct_agg":"c_b",
                                  "appear_in_n_funds":"appear_in_n_funds_b"}))
            j=pd.merge(xa,xb,on=["ticker","holding_name"],how="outer")
            j["c_a"]=j["c_a"].fillna(0.0);j["c_b"]=j["c_b"].fillna(0.0)
            j["delta_contrib_pct_agg"]=j["c_b"]-j["c_a"]
            j["year_a"]=a;j["year_b"]=ref_year
            pairs.append(j[["year_a","year_b","holding_name","ticker","classification_a","classification_b","delta_contrib_pct_agg","appear_in_n_funds_a","appear_in_n_funds_b"]])
    movers=pd.concat(pairs,ignore_index=True) if pairs else pd.DataFrame(columns=["year_a","year_b","holding_name","ticker","classification_a","classification_b","delta_contrib_pct_agg","appear_in_n_funds_a","appear_in_n_funds_b"])
    movers.sort_values(["year_a","delta_contrib_pct_agg"],ascending=[True,False]).to_csv(OUT_MOVERS,index=False)

if __name__=="__main__":
    build_exposures_by_fund_year()
    build_aggregates()
    build_dispersion()
    build_screens_and_movers()
    print("Analysis 2 complete.")

/var/folders/11/2mby0xs907bb9d9x_zhhsyn00000gn/T/ipykernel_4516/1224600571.py:177: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  aum_true=df2.groupby(year_col,group_keys=False).apply(wavg).reset_index()


Analysis 2 complete.


In [37]:
# Anlaysis 2 - ADD ON

import pandas as pd
import os
import re

BASE_DIR = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 2"
MOVERS_CSV = os.path.join(BASE_DIR, "movers_by_yearpair.csv")
HOLD_2025 = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data/holdings_2025_final.csv"
OUT_TOP = os.path.join(BASE_DIR, "top_movers_with_names.csv")

def canon_ticker(t):
    if pd.isna(t): return t
    s=str(t).strip().upper().replace("/",".").replace("-",".").replace(" ",".")
    s=re.sub(r"\.+",".",s)
    return s

def canon_name(s):
    if pd.isna(s): return s
    s=str(s).lower().replace("&"," and ")
    s=re.sub(r"[^a-z0-9 ]+"," ",s)
    s=re.sub(r"\b(inc|inc\.|corp|corporation|plc|ltd|llc|co|co\.|company|sa|nv|ag|spa|s\.p\.a|holdings|group|limited)\b","",s)
    s=re.sub(r"\bclass\s+[abc]\b","",s)
    s=re.sub(r"\s+"," ",s).strip()
    return s

movers = pd.read_csv(MOVERS_CSV)
hold25 = pd.read_csv(HOLD_2025)

movers.columns = movers.columns.str.strip()
hold25.columns = hold25.columns.str.strip()

movers["ticker"] = movers["ticker"].astype(str).str.upper().str.strip().map(canon_ticker)
movers["holding_name"] = movers["holding_name"].astype(str).str.strip()
movers["holding_name_key"] = movers["holding_name"].map(canon_name)

# accept both old and new schemas gracefully
cls_a_col = "classification_a" if "classification_a" in movers.columns else "classification"
cls_b_col = "classification_b" if "classification_b" in movers.columns else "classification"

# 2025 name columns (handle variants)
c_ticker = next(c for c in hold25.columns if c.lower() in {"company_ticker","ticker","symbol"})
name_og = next(c for c in hold25.columns if c.lower() in {"name","security_name","holding_name"})
name_norm = next(c for c in hold25.columns if c.lower() in {"name_normalized","normalised name","normalised_name","normalized_name","normalised_name"})

hold25["company_ticker"] = hold25[c_ticker].astype(str).str.upper().str.strip().map(canon_ticker)
hold25["Name"] = hold25[name_og].astype(str).str.strip()
hold25["name_normalized"] = hold25[name_norm].astype(str).str.strip()
hold25["name_key_2025"] = hold25["name_normalized"].map(canon_name)

agg = (
    movers.groupby(["year_a","year_b","ticker","holding_name"], as_index=False)
    .agg({
        "delta_contrib_pct_agg":"sum",
        "appear_in_n_funds_a":"mean",
        "appear_in_n_funds_b":"mean",
        cls_a_col: lambda x: x.dropna().iloc[0] if x.dropna().size else None,
        cls_b_col: lambda x: x.dropna().iloc[0] if x.dropna().size else None,
    })
)

agg = agg.drop_duplicates(subset=["year_a","year_b","ticker","holding_name"])
agg["holding_name_key"] = agg["holding_name"].map(canon_name)

top_list = []
for y in sorted(agg["year_a"].dropna().unique()):
    sub = agg[agg["year_a"] == y].copy()
    sub = sub.sort_values("delta_contrib_pct_agg", ascending=False).head(10)
    sub["rank_within_year"] = range(1, len(sub) + 1)
    top_list.append(sub)
top_all = pd.concat(top_list, ignore_index=True) if top_list else agg.head(0)

m1 = top_all.merge(
    hold25[["company_ticker","Name"]],
    how="left",
    left_on="ticker",
    right_on="company_ticker"
).rename(columns={"Name":"Name_2025"})

m2 = m1.merge(
    hold25[["name_key_2025","Name"]].drop_duplicates("name_key_2025"),
    how="left",
    left_on="holding_name_key",
    right_on="name_key_2025",
    suffixes=("","_byname")
)

m2["Name_2025"] = m2["Name_2025"].where(m2["Name_2025"].notna(), m2["Name"])
out_cols = [
    "year_a","year_b","rank_within_year","ticker","holding_name","Name_2025",
    cls_a_col, cls_b_col, "delta_contrib_pct_agg","appear_in_n_funds_a","appear_in_n_funds_b"
]

m2[out_cols].drop_duplicates(subset=["year_a","ticker"]).to_csv(OUT_TOP, index=False)
print("Final top movers cleaned and saved to:", OUT_TOP)


Final top movers cleaned and saved to: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 2/top_movers_with_names.csv


In [25]:
# Analysis 3 
import os, re
import pandas as pd
import numpy as np

BASE_DIR  = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data"
PRICES_DIR= "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data/Prices"
OUT_DIR   = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 3"

HOLDINGS_2025_CSV = os.path.join(BASE_DIR, "holdings_2025_final.csv")
PRICES_XLSX       = os.path.join(PRICES_DIR, "holdings_2025_prices_final.xlsx")
CLASS_BIN_CSV     = os.path.join(BASE_DIR, "classification_binary.csv")
OUT_XLSX          = os.path.join(OUT_DIR, "universe_2025.xlsx")

def pick_col(cols, names):
    """Return the first column that matches any of the candidate names (case/space/punct-insensitive)."""
    low = {c.lower(): c for c in cols}
    for n in names:
        key = n.lower()
        if key in low:
            return low[key]
    def norm(s): return re.sub(r"[^a-z0-9]", "", s.lower())
    norm_map = {norm(c): c for c in cols}
    for n in names:
        nn = norm(n)
        if nn in norm_map:
            return norm_map[nn]
    for c in cols:
        cc = norm(c)
        for n in names:
            if "|" in n:
                parts = [norm(x) for x in n.split("|")]
                if all(p in cc for p in parts):
                    return c
    return None

def require(col, label):
    if col is None:
        raise ValueError(f"Required column missing for: {label}")

def norm_name(s):
    if pd.isna(s): return None
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

h = pd.read_csv(HOLDINGS_2025_CSV, dtype=str)
for c in h.columns:
    if h[c].dtype == object:
        h[c] = h[c].astype(str).str.strip()

etf_col = pick_col(h.columns, ["ETF_Ticker","ETF Ticker","etf_ticker","fund","etf"])
tic_col = pick_col(h.columns, ["company_ticker","ticker","symbol"])
nn_col  = pick_col(h.columns, ["name_normalised","name_normalized","company_name","name"])
sec_col = pick_col(h.columns, ["Sector"])
loc_col = pick_col(h.columns, ["Location","Country","Region"])
w_col   = pick_col(h.columns, ["Weight (%)","weight (%)","Weight%","weight","portfolio_weight_pct"])
mv_col  = pick_col(h.columns, ["Market Value","market value","market_value_usd","market_value"])

require(etf_col, "ETF_Ticker")
require(tic_col, "company_ticker")
require(nn_col,  "name_normalized")
require(sec_col, "Sector")
require(loc_col, "Location/Region")
require(w_col,   "Weight (%)")
require(mv_col,  "Market Value")

base = h[[etf_col, tic_col, nn_col, sec_col, loc_col, w_col, mv_col]].copy()
base.columns = ["ETF_Ticker","company_ticker","name_normalized","Sector","Location","Weight (%)","Market Value"]
base["_row_order"] = np.arange(len(base))
base["_name_key"] = base["name_normalized"].map(norm_name)

p = pd.read_excel(PRICES_XLSX, dtype=str)
for c in p.columns:
    if p[c].dtype == object:
        p[c] = p[c].astype(str).str.strip()

p_name  = pick_col(p.columns, ["company_name","name","name_normalized","name_normalised"])
p_price = pick_col(p.columns, ["price","price_usd","usd_price","price usd"])

require(p_name,  "prices.company_name")
require(p_price, "prices.price")

p_map = p[[p_name, p_price]].drop_duplicates().rename(
    columns={p_name:"company_name", p_price:"price"}
)
p_map["_name_key"] = p_map["company_name"].map(norm_name)

m_prices = base.merge(p_map[["_name_key","price"]], how="left", on="_name_key")

if m_prices["price"].isna().any():
    p_tic = pick_col(p.columns, ["ticker","symbol","company_ticker"])
    if p_tic:
        p_map2 = p[[p_tic, p_price]].drop_duplicates().rename(
            columns={p_tic:"company_ticker", p_price:"price_by_ticker"}
        )
        m_prices = m_prices.merge(p_map2, how="left", on="company_ticker")
        m_prices["price"] = m_prices["price"].fillna(m_prices["price_by_ticker"])
        m_prices = m_prices.drop(columns=["price_by_ticker"], errors="ignore")

cb = pd.read_csv(CLASS_BIN_CSV, dtype=str)
for c in cb.columns:
    if cb[c].dtype == object:
        cb[c] = cb[c].astype(str).str.strip()

cb_etf  = pick_col(cb.columns, ["ETF Ticker","ETF_Ticker","etf_ticker","fund","etf"])
cb_tic  = pick_col(cb.columns, ["company_ticker","ticker","symbol"])
cb_name = pick_col(cb.columns, ["name_normalized","name_normalised","name","company_name"])
cb_c200 = pick_col(cb.columns, ["clean200","clean_200"])
cb_def  = pick_col(cb.columns, ["deforestation"])
cb_fos  = pick_col(cb.columns, ["fossil fuel","fossil fuels","fossilfuel","fossil_fuel"])
cb_pri  = pick_col(cb.columns, ["prison","prisons","private prisons","privateprisons","private_prisons"])
cb_tob  = pick_col(cb.columns, ["tobacco"])
cb_wep  = pick_col(cb.columns, ["weapons","weapon","controversial weapons","controversialweapons","controversial_weapons"])

require(cb_name, "classification.name")

flag_cols = [cb_c200, cb_fos, cb_tob, cb_wep, cb_pri, cb_def]
flag_names= ["Clean200","FossilFuel","Tobacco","Weapons","Prisons","Deforestation"]

cb_flags = cb[[c for c in [cb_etf, cb_tic, cb_name] if c is not None] + [c for c in flag_cols if c is not None]].copy()
rename_map = {}
if cb_etf:  rename_map[cb_etf] = "ETF_Ticker"
if cb_tic:  rename_map[cb_tic] = "company_ticker"
rename_map[cb_name] = "name_normalized"
cb_flags = cb_flags.rename(columns=rename_map)

cb_flags["_name_key"] = cb_flags["name_normalized"].map(norm_name)

for src, tgt in zip(flag_cols, flag_names):
    if src and src in cb_flags.columns:
        cb_flags[tgt] = cb_flags[src]
    if tgt not in cb_flags.columns:
        cb_flags[tgt] = 0
    cb_flags[tgt] = cb_flags[tgt].map({"1":1,"0":0,"Y":1,"N":0,"Yes":1,"No":0,1:1,0:0}).fillna(0).astype(int)

group_keys = [k for k in ["ETF_Ticker","company_ticker","name_normalized"] if k in cb_flags.columns]
cb_agg = cb_flags.groupby(group_keys, dropna=False)[flag_names].max().reset_index()
cb_agg["_name_key"] = cb_agg["name_normalized"].map(norm_name)

m_cls = m_prices.copy()

if all(k in m_cls.columns for k in ["ETF_Ticker","company_ticker","name_normalized"]) and \
   all(k in cb_agg.columns for k in ["ETF_Ticker","company_ticker","name_normalized"]):
    m_cls = m_cls.merge(
        cb_agg[["ETF_Ticker","company_ticker","name_normalized"] + flag_names],
        how="left",
        on=["ETF_Ticker","company_ticker","name_normalized"],
        suffixes=("","")
    )

missing_mask = m_cls[flag_names].isna().all(axis=1)
if missing_mask.any():
    fill = m_cls.loc[missing_mask, ["_name_key"]].merge(
        cb_agg[["_name_key"] + flag_names].drop_duplicates("_name_key"),
        how="left",
        on="_name_key"
    )
    for col in flag_names:
        m_cls.loc[missing_mask, col] = m_cls.loc[missing_mask, col].fillna(fill[col])

for col in flag_names:
    if col not in m_cls.columns:
        m_cls[col] = 0
    m_cls[col] = m_cls[col].fillna(0).astype(int)

m_cls["Controversies"] = ((m_cls["FossilFuel"] + m_cls["Tobacco"] + m_cls["Weapons"] + m_cls["Prisons"] + m_cls["Deforestation"]) > 0).astype(int)
m_cls["Clean"] = (m_cls["Clean200"] > 0).astype(int)

m_cls = m_cls.sort_values("_row_order")

m_cls["company_name"] = m_cls.get("company_name", m_cls["name_normalized"])

os.makedirs(OUT_DIR, exist_ok=True)
out_cols = [
    "ETF_Ticker","company_ticker","company_name","name_normalized",
    "Sector","Location","Weight (%)","Market Value","price",
    "Clean200","FossilFuel","Tobacco","Weapons","Prisons","Deforestation",
    "Controversies","Clean"
]
out = m_cls[out_cols].copy()
out.to_excel(OUT_XLSX, index=False)
print(f"Wrote: {OUT_XLSX}  rows={len(out):,}")


Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 3/universe_2025.xlsx  rows=8,249


In [26]:
#Analysis 3 

#!/usr/bin/env python3
import os, re, time
import pandas as pd
import numpy as np
from tqdm import tqdm

UNIVERSE_PATH = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 3/universe_2025.xlsx"
OUT_DIR = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 3"
OUT_ALL = os.path.join(OUT_DIR, "returns_top_per_etf_2025.csv")

TOP_PER_ETF = 20
COVERAGE_FLOOR = None
PERIOD = "6mo"
BATCH_SIZE = 80
SLEEP_SEC = 1.0
WINSOR_LO_HI = (0.01, 0.99)

def ensure_dir(p): os.makedirs(p, exist_ok=True)

def pick_col(cols, candidates):
    cmap = {c.lower(): c for c in cols}
    for c in candidates:
        if c.lower() in cmap: return cmap[c.lower()]
    def simp(x): return re.sub(r"[^a-z0-9]", "", x.lower())
    smap = {simp(c): c for c in cols}
    for c in candidates:
        sc = simp(c)
        if sc in smap: return smap[sc]
    for col in cols:
        cc = simp(col)
        for c in candidates:
            parts = [simp(p) for p in c.split("|")]
            if all(p in cc for p in parts):
                return col
    return None

def clean_weight(series):
    s = pd.to_numeric(series, errors="coerce")
    if s.dropna().between(0,1).mean() > 0.8 and s.max() <= 1.5: s = s * 100.0
    return s

SUFFIX_MAP = {
    "united states":"", "usa":"", "us":"", "u.s.":"", "u.s":"",
    "canada":".TO", "brazil":".SA",
    "united kingdom":".L", "uk":".L", "great britain":".L", "england":".L",
    "germany":".DE", "france":".PA", "netherlands":".AS", "belgium":".BR",
    "italy":".MI", "spain":".MC", "portugal":".LS", "sweden":".ST",
    "denmark":".CO", "norway":".OL", "finland":".HE", "switzerland":".SW",
    "austria":".VI", "ireland":".IR",
    "japan":".T", "hong kong":".HK", "china":".SS", "taiwan":".TW",
    "south korea":".KS", "korea":".KS", "india":".NS", "australia":".AX",
    "singapore":".SI", "new zealand":".NZ",
    "europe":".PA", "emea":".PA", "asia":".T", "apac":".T"
}

MANUAL_MAP = {
    "BRK.B":"BRK-B", "BF.B":"BF-B",
    "NOVO B":"NOVO-B.CO", "NOVOB":"NOVO-B.CO", "NOVO-B":"NOVO-B.CO",
    "ORSTED":"ORSTED.CO", "VWS":"VWS.CO",
    "NOVN":"NOVN.SW", "NESN":"NESN.SW", "UBSG":"UBSG.SW", "ZURN":"ZURN.SW", "SREN":"SREN.SW", "GIVN":"GIVN.SW",
    "HSBA":"HSBA.L", "LLOY":"LLOY.L", "BARC":"BARC.L", "DGE":"DGE.L", "REL":"REL.L", "LR":"LR.PA",
    "SIE":"SIE.DE", "IFX":"IFX.DE", "DB1":"DB1.DE", "NDX1":"NDX1.DE", "LONN":"LONN.SW",
    "IBE":"IBE.MC", "EDP":"EDP.LS", "EDPR":"EDPR.LS",
    "UCG":"UCG.MI", "INGA":"INGA.AS", "PRX":"PRX.AS",
    "700":"0700.HK", "968":"0968.HK", "388":"0388.HK", "2318":"2318.HK", "2015":"2015.HK", "9988":"9988.HK",
    "3690":"3690.HK", "9868":"9868.HK", "1299":"1299.HK", "1928":"1928.HK", "1120":"1120.HK", "66":"0066.HK",
    "2330":"2330.TW",
    "RELIANCE":"RELIANCE.NS", "HDFCBANK":"HDFCBANK.NS", "HINDUNILVR":"HINDUNILVR.NS", "NHPC":"NHPC.NS", "SUZLON":"SUZLON.NS",
    "600900":"600900.SS",
    "NPI":"NPI.TO", "WSP":"WSP.TO", "MOWI":"MOWI.OL",
    "XTSLA": None, "9434-T.T":"9434.T",
    "288.HK":"0288.HK", "660":"0660.HK", "268.SS":"0268.HK", "9999.SS":"9999.HK",
    "3988.SS":"3988.HK", "2269.SS":"2269.HK",
    "IDCBY.SS":"IDCBY", "SPOT.ST":"SPOT", "ESSITYB.ST":"ESSITY-B.ST", "373220":"373220.KS",
    "NPN":"NPN.JO"
}

def normalize_tkr(t):
    if t is None or (isinstance(t,float) and pd.isna(t)): return None
    return str(t).strip().upper().replace("  "," ")

def guess_yahoo_symbol(ticker, location):
    t0 = normalize_tkr(ticker)
    if not t0: return None, "empty_ticker"
    if t0 in MANUAL_MAP: return MANUAL_MAP[t0], "manual_map"
    t_cls = t0.replace(".", "-").replace(" ", "")
    loc = (location or "").strip().lower()
    suffix = SUFFIX_MAP.get(loc, "")
    return f"{t_cls}{suffix}", f"suffix_by_location({loc or 'unknown'})"

def winsorize(s, lo=0.01, hi=0.99):
    a, b = s.quantile(lo), s.quantile(hi)
    return s.clip(a, b)

def select_top_per_etf(uni, c_etf, c_tic, c_w, c_loc):
    df = uni.copy()
    df["_etf"] = df[c_etf].astype(str).str.strip().str.upper()
    df["_tic"] = df[c_tic].astype(str).str.strip().str.upper()
    df["_w"] = clean_weight(df[c_w])
    if c_loc: df["_loc"] = df[c_loc].astype(str)
    else: df["_loc"] = pd.Series(["United States"] * len(df), index=df.index)
    selected = []
    for etf, g in df.groupby("_etf", sort=False):
        g2 = g.sort_values("_w", ascending=False).reset_index(drop=True)
        if COVERAGE_FLOOR:
            g2["_cum"] = g2["_w"].fillna(0).cumsum() / g2["_w"].fillna(0).sum()
            keep = g2[g2["_cum"] <= float(COVERAGE_FLOOR)]
        else:
            keep = g2.head(int(TOP_PER_ETF))
        selected.append(keep[["_etf","_tic","_w","_loc"]])
    sel = pd.concat(selected, ignore_index=True)
    sel["_key"] = sel["_tic"]
    agg = sel.sort_values("_w", ascending=False).drop_duplicates("_key")
    etfs_by_ticker = sel.groupby("_tic")["_etf"].apply(lambda s: ";".join(sorted(set(s)))).reset_index().rename(columns={"_tic":"ticker","_etf":"etfs"})
    agg = agg.rename(columns={"_tic":"ticker","_w":"agg_weight_pct","_loc":"location"}).drop(columns=["_key"])
    out = agg.merge(etfs_by_ticker, on="ticker", how="left")
    return out

def main():
    import yfinance as yf
    ensure_dir(OUT_DIR)
    uni = pd.read_excel(UNIVERSE_PATH) if UNIVERSE_PATH.lower().endswith((".xlsx",".xls")) else pd.read_csv(UNIVERSE_PATH)
    cols = list(uni.columns)
    c_etf = pick_col(cols, ["ETF_Ticker","etf","fund"])
    c_tic = pick_col(cols, ["company_ticker","ticker","symbol"])
    c_w = pick_col(cols, ["Weight (%)","weight (%)","Weight%","weight","portfolio_weight_pct"])
    c_loc = pick_col(cols, ["Location","Country","Region"])
    if not (c_etf and c_tic and c_w): raise ValueError("universe must have ETF, ticker, and weight columns.")
    top = select_top_per_etf(uni, c_etf, c_tic, c_w, c_loc)
    rows = []
    for _, r in top.iterrows():
        sym, note = guess_yahoo_symbol(r["ticker"], r.get("location"))
        rows.append({**r.to_dict(), "yahoo_symbol": sym, "notes": note})
    ymap = pd.DataFrame(rows)
    symbols = [s for s in ymap["yahoo_symbol"].dropna().tolist() if s]
    frames = []
    for i in tqdm(range(0, len(symbols), BATCH_SIZE), desc="Downloading"):
        batch = symbols[i:i+BATCH_SIZE]
        try:
            data = yf.download(batch, period=PERIOD, interval="1d", auto_adjust=True, progress=False, group_by="ticker", threads=True)
        except Exception:
            time.sleep(SLEEP_SEC); continue
        if data is None or data.empty:
            time.sleep(SLEEP_SEC); continue
        if isinstance(data.columns, pd.MultiIndex):
            for sym in batch:
                try: sub = data[sym]
                except KeyError: continue
                pick = None
                for cname in sub.columns:
                    c0 = cname.lower().replace(" ", "")
                    if c0 in ("adjclose","close","price"): pick = cname; break
                if pick is None: continue
                px = sub[pick].dropna()
                if px.empty: continue
                dfp = px.to_frame("px").reset_index()
                dfp.rename(columns={dfp.columns[0]:"date"}, inplace=True)
                dfp["ticker_yahoo"] = sym
                frames.append(dfp[["date","ticker_yahoo","px"]])
        else:
            cols_lower = [c.lower().replace(" ","") for c in data.columns]
            pick = None
            for cname, c0 in zip(data.columns, cols_lower):
                if c0 in ("adjclose","close","price"): pick = cname; break
            if pick is None: pass
            else:
                px = data[pick].dropna()
                if not px.empty:
                    dfp = px.to_frame("px").reset_index()
                    dfp.rename(columns={dfp.columns[0]:"date"}, inplace=True)
                    dfp["ticker_yahoo"] = batch[0]
                    frames.append(dfp[["date","ticker_yahoo","px"]])
        time.sleep(SLEEP_SEC)
    if not frames: raise RuntimeError("No price data downloaded.")
    prices = pd.concat(frames, ignore_index=True)
    prices["date"] = pd.to_datetime(prices["date"], errors="coerce")
    prices = prices.dropna(subset=["date"]).sort_values(["ticker_yahoo","date"])
    prices["ret"] = prices.groupby("ticker_yahoo")["px"].pct_change()
    prices = prices.dropna(subset=["ret"])
    if WINSOR_LO_HI:
        lo, hi = WINSOR_LO_HI
        prices["ret"] = prices.groupby("ticker_yahoo")["ret"].transform(lambda s: s.clip(s.quantile(lo), s.quantile(hi)))
    rev = dict(zip(ymap["yahoo_symbol"], ymap["ticker"]))
    prices["ticker"] = prices["ticker_yahoo"].map(rev)
    prices = prices.dropna(subset=["ticker"])
    meta = ymap.set_index("ticker")[["etfs","location","agg_weight_pct","yahoo_symbol"]]
    out = prices.merge(meta, on="ticker", how="left")[["date","ticker","ret","etfs","location","agg_weight_pct","yahoo_symbol"]]
    out = out.sort_values(["ticker","date"])
    out.to_csv(OUT_ALL, index=False)
    print(f"[done] Wrote {OUT_ALL}  rows={len(out):,}  tickers={out['ticker'].nunique()}")

if __name__ == "__main__":
    main()


Downloading:   0%|                                        | 0/3 [00:00<?, ?it/s]HTTP Error 404: 
HTTP Error 404: 

7 Failed downloads:
['BRKB', '288.HK', 'ESSITYB.ST', '939.SS', '268.SS', '5930', '1810.SS']: YFPricesMissingError('possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")')
Downloading:  33%|██████████▋                     | 1/3 [00:10<00:20, 10.28s/it]
8 Failed downloads:
['1398.SS', '3988.SS', 'SPOT.ST', '9618.SS', '9961.SS', 'NU.SA', '9999.SS', '316140']: YFPricesMissingError('possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")')
Downloading:  67%|█████████████████████▎          | 2/3 [00:19<00:09,  9.44s/it]
1 Failed download:
['105560']: YFPricesMissingError('possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")')
Downloading: 100%|████████████████████████████████| 3/3 [00:22<00:00,  7.48s/it]


[done] Wrote /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 3/returns_top_per_etf_2025.csv  rows=22,029  tickers=176


In [27]:
#Analysis 3
import os
import pandas as pd
import numpy as np

BASE_DIR = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data"
FINAL_DIR = os.path.join(BASE_DIR, "Final Data")
PRICES_DIR = os.path.join(FINAL_DIR, "Prices")
DASH_DIR = os.path.join(BASE_DIR, "Data for Dashboard", "Analysis 3")
os.makedirs(DASH_DIR, exist_ok=True)

UNIVERSE = os.path.join(DASH_DIR, "universe_2025.xlsx")
RETURNS = os.path.join(DASH_DIR, "returns_top_per_etf_2025.csv")
COV_OUT = os.path.join(DASH_DIR, "covariance_2025.csv")

if not os.path.exists(UNIVERSE):
    raise FileNotFoundError("universe_2025.xlsx not found in Analysis 3 folder")
if not os.path.exists(RETURNS):
    raise FileNotFoundError("returns_top_per_etf_2025.csv not found in Analysis 3 folder")

r = pd.read_csv(RETURNS)
r.columns = [c.lower() for c in r.columns]
req = {"date", "ticker", "ret"}
if not req.issubset(set(r.columns)):
    raise ValueError(f"Missing required columns in returns file: {req - set(r.columns)}")

r_pivot = r.pivot(index="date", columns="ticker", values="ret").fillna(0.0)
cov = r_pivot.cov()
if isinstance(cov, pd.Series):
    cov = cov.to_frame()

cov.to_csv(COV_OUT, float_format="%.10f")
print(f"Saved covariance matrix to: {COV_OUT}")


Saved covariance matrix to: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 3/covariance_2025.csv


In [28]:
#Analysis 3

import os, json, math, re
import pandas as pd
import numpy as np
from datetime import datetime
from time import perf_counter

UNIVERSE_PATH = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 3/universe_2025.xlsx"
COV_PATH = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 3/covariance_2025.csv"
OUT_DIR = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 3"
PORT_PATH = os.path.join(OUT_DIR, "scenario_portfolio_metrics.csv")
POS_PATH = os.path.join(OUT_DIR, "scenario_position_deltas.csv")
PROG_PATH = os.path.join(OUT_DIR, "scenario_progress.csv")
VERBOSE = True

SCENARIOS = ["Baseline","PragmaticTilt","StrictExclusion"]
TE_BUDGET_TILT = 0.02
SECTOR_BAND = 0.02
REGION_BAND = 0.02
CAP_ABS = 0.05
CAP_X_BASE = 3.0
CLEAN_TILT_TARGET_PP = 30
EPS_SHIFT = 0.001
HEARTBEAT_EVERY = 200
ETF_TIMEOUT_S = 60
MAX_ITERS = 100000

def pick(cols, cands):
    low = {c.lower(): c for c in cols}
    for n in cands:
        if n.lower() in low: return low[n.lower()]
    def norm(s): return re.sub(r"[^a-z0-9]", "", s.lower())
    nm = {norm(c): c for c in cols}
    for n in cands:
        k = norm(n)
        if k in nm: return nm[k]
    for c in cols:
        cc = norm(c)
        for n in cands:
            parts = [norm(x) for x in n.split("|")]
            if all(p in cc for p in parts): return c
    return None

def log_progress(scenario_id, etf, stage, start_t, extra=None):
    row = {
        "timestamp": datetime.now().isoformat(),
        "scenario_id": scenario_id,
        "etf": etf if etf is not None else "",
        "stage": stage,
        "elapsed_s": round(perf_counter()-start_t, 4)
    }
    if isinstance(extra, dict): row.update(extra)
    df = pd.DataFrame([row])
    mode = "a" if os.path.exists(PROG_PATH) else "w"
    header = not os.path.exists(PROG_PATH)
    df.to_csv(PROG_PATH, mode=mode, header=header, index=False)
    if VERBOSE:
        extra_txt = "" if extra is None else " " + str(extra)
        print(f"[{row['timestamp']}] {scenario_id} | {row['etf']} | {stage} | {row['elapsed_s']}s{extra_txt}")

def load_universe(path):
    u = pd.read_excel(path)
    etf = pick(u.columns, ["ETF_Ticker","etf","fund"])
    tick = pick(u.columns, ["company_ticker","ticker","symbol"])
    name = pick(u.columns, ["name_normalized","name_normalised","company_name","name"])
    sector = pick(u.columns, ["Sector"])
    region = pick(u.columns, ["Region","Location","Country"])
    wcol = pick(u.columns, ["Weight (%)","weight_pct","weight","portfolio_weight_pct"])
    clean = pick(u.columns, ["Clean","Clean200"])
    fos = pick(u.columns, ["FossilFuel","fossil"])
    tob = pick(u.columns, ["Tobacco"])
    wep = pick(u.columns, ["Weapons","weapon"])
    pri = pick(u.columns, ["Prisons","prison"])
    defo = pick(u.columns, ["Deforestation","deforest"])
    if etf is None or tick is None or name is None or wcol is None or sector is None or region is None:
        raise ValueError("Universe missing required columns")
    for c in [clean,fos,tob,wep,pri,defo]:
        if c and u[c].dtype!=int:
            u[c] = pd.to_numeric(u[c], errors="coerce").fillna(0).astype(int)
    u = u.rename(columns={etf:"ETF_Ticker",tick:"Ticker",name:"Name",sector:"Sector",region:"Region",wcol:"Weight"})
    if clean is None: u["Clean"]=0
    else: u = u.rename(columns={clean:"Clean"})
    if fos is None: u["FossilFuel"]=0
    else: u = u.rename(columns={fos:"FossilFuel"})
    if tob is None: u["Tobacco"]=0
    else: u = u.rename(columns={tob:"Tobacco"})
    if wep is None: u["Weapons"]=0
    else: u = u.rename(columns={wep:"Weapons"})
    if pri is None: u["Prisons"]=0
    else: u = u.rename(columns={pri:"Prisons"})
    if defo is None: u["Deforestation"]=0
    else: u = u.rename(columns={defo:"Deforestation"})
    u["Controversial"] = ((u["FossilFuel"]+u["Tobacco"]+u["Weapons"]+u["Prisons"]+u["Deforestation"])>0).astype(int)
    u["Weight"] = pd.to_numeric(u["Weight"], errors="coerce").fillna(0.0)
    return u

def load_cov(path):
    df = pd.read_csv(path)
    if df.shape[1]-1==df.shape[0] and not np.issubdtype(df.iloc[:,0].dtype, np.number):
        idx = df.columns[0]
        df = df.set_index(idx)
    if df.index.name is None or any(df.index!=df.columns):
        try:
            df = df.set_index(df.columns[0])
            df = df.loc[df.columns, df.columns]
        except:
            pass
    df = df.apply(pd.to_numeric, errors="coerce").fillna(0.0)
    return df

def comp_split(df, w):
    c = (df["Clean"]*w).sum()
    x = (df["Controversial"]*w).sum()
    o = w.sum()-c-x
    tot = w.sum()
    return 100*c/tot if tot>0 else 0, 100*x/tot if tot>0 else 0, 100*o/tot if tot>0 else 0

def active_share(w0, w1): return 50.0*np.abs(w1-w0).sum()

def te_annual(w0, w1, cov, tickers):
    dw = (w1 - w0).values
    tks = tickers.values
    sub = [t for t in tks if t in cov.columns]
    if len(sub)==0: return np.nan
    idx = [np.where(tks==t)[0][0] for t in sub]
    dw_sub = dw[idx]
    C = cov.loc[sub, sub].values
    val = float(np.dot(dw_sub, np.dot(C, dw_sub)))
    if val<0: val=0.0
    return math.sqrt(val)*math.sqrt(252.0)*100.0

def te_state_init(w0, w, cov, tickers):
    tks = tickers.values
    sub = [t for t in tks if t in cov.columns]
    if len(sub)==0:
        return None
    idx = np.array([np.where(tks==t)[0][0] for t in sub], dtype=int)
    C = cov.loc[sub, sub].values
    dw = (w - w0).values[idx]
    y = C @ dw
    var = float(dw @ y)
    return {"idx": idx, "C": C, "dw": dw, "y": y, "var": var}

def te_try_increment(state, i_all, j_all, inc, w0, w, tickers, te_budget):
    if state is None:
        return True
    idx = state["idx"]; C = state["C"]
    dw = state["dw"]; y = state["y"]; var = state["var"]
    all2sub = {v:k for k,v in enumerate(idx)}
    if i_all not in all2sub or j_all not in all2sub:
        return True
    i = all2sub[i_all]; j = all2sub[j_all]
    delta_var = -2.0*inc*y[i] + 2.0*inc*y[j] + inc*inc*(C[i,i] + C[j,j] - 2.0*C[i,j])
    var_new = var + delta_var
    if var_new < 0: var_new = 0.0
    te_annual_pct = math.sqrt(var_new)*math.sqrt(252.0)*100.0
    if te_annual_pct <= te_budget*100.0:
        dw[i] -= inc; dw[j] += inc
        y[:] = y - inc*C[:, i] + inc*C[:, j]
        state["var"] = var_new
        return True
    return False

def te_current_pct(state):
    if state is None:
        return np.nan
    var = max(state["var"], 0.0)
    return math.sqrt(var)*math.sqrt(252.0)*100.0

def redistribute_within(df, w, mask_allowed, cap, sector_target, region_target):
    w = w.copy()
    for _ in range(100000):
        if abs(w.sum()-1.0)<1e-9: break
        if w.sum()<1.0:
            pool = min(EPS_SHIFT, 1.0 - w.sum())
            elig = mask_allowed & (w<cap)
            if not elig.any(): break
            sec_cur = (df.assign(w=w).groupby("Sector")["w"].sum())
            reg_cur = (df.assign(w=w).groupby("Region")["w"].sum())
            sec_cur = sec_cur/sec_cur.sum() if sec_cur.sum()>0 else sec_cur
            reg_cur = reg_cur/reg_cur.sum() if reg_cur.sum()>0 else reg_cur
            scores = []
            for i in np.where(elig)[0]:
                s = df.iloc[i]["Sector"]; r = df.iloc[i]["Region"]
                s_gap = sector_target.get(s,0)-sec_cur.get(s,0)
                r_gap = region_target.get(r,0)-reg_cur.get(r,0)
                scores.append((i, max(s_gap,0)+max(r_gap,0)))
            if not scores: break
            scores.sort(key=lambda x: x[1], reverse=True)
            i = scores[0][0]
            w.iloc[i] = min(cap.iloc[i], w.iloc[i]+pool)
        else:
            pool = min(EPS_SHIFT, w.sum()-1.0)
            elig = w>0
            i = np.where(elig)[0][0]
            w.iloc[i] = max(0.0, w.iloc[i]-pool)
    return w

def pragmatic_tilt(scenario_id, etf, df, cov, te_budget, t_etf):
    w0 = pd.Series(df["Weight"].values/100.0, index=df.index)
    w  = w0.copy()
    cap = pd.Series(np.minimum(CAP_ABS, CAP_X_BASE*(w0+1e-12)), index=df.index)
    clean_mask = (df["Clean"]==1)
    bad_mask   = (df["Controversial"]==1)
    other_mask = ~(clean_mask|bad_mask)
    order_src = np.argsort(-bad_mask.astype(int)*w.values - 0.5*other_mask.astype(int)*w.values)
    order_dst = np.argsort(-(clean_mask.astype(int))*(cap - w).clip(lower=0).values)
    tickers = df["Ticker"]
    state = te_state_init(w0, w, cov, tickers)
    te_calls = 0
    moves = 0
    iters = 0
    last_moves = 0
    while iters < MAX_ITERS:
        iters += 1
        moved = False
        for si in order_src:
            if w.iloc[si] <= 0:
                continue
            src_sector = df.iloc[si]["Sector"]
            src_region = df.iloc[si]["Region"]
            src_pool   = min(EPS_SHIFT, w.iloc[si])
            if src_pool <= 0:
                continue
            for di in order_dst:
                if not clean_mask.iloc[di]:
                    continue
                if cap.iloc[di] - w.iloc[di] <= 1e-12:
                    continue
                if df.iloc[di]["Sector"]!=src_sector and df.iloc[di]["Region"]!=src_region:
                    continue
                inc = min(src_pool, cap.iloc[di]-w.iloc[di])
                te_calls += 1
                if te_try_increment(state, si, di, inc, w0, w, tickers, te_budget):
                    w.iloc[si] -= inc
                    w.iloc[di] += inc
                    moves += 1
                    moved = True
                    break
            if moved:
                break
        if (iters % HEARTBEAT_EVERY) == 0:
            pc, px, _ = comp_split(df, w)
            te_now = te_current_pct(state)
            log_progress(scenario_id, etf, "tilt_heartbeat", t_etf, {"iters": iters, "moves": moves, "te_calls": te_calls, "pct_clean": round(pc,2), "pct_contro": round(px,2), "te_pct": (None if np.isnan(te_now) else round(te_now,3))})
            if moves == last_moves:
                log_progress(scenario_id, etf, "tilt_stall_hint", t_etf, {"hint":"no_moves_since_last_hb"})
            last_moves = moves
        if not moved:
            te_now = te_current_pct(state)
            log_progress(scenario_id, etf, "tilt_converged", t_etf, {"iters": iters, "moves": moves, "te_calls": te_calls, "te_pct": (None if np.isnan(te_now) else round(te_now,3))})
            break
        if perf_counter() - t_etf > ETF_TIMEOUT_S:
            pc, px, _ = comp_split(df, w)
            te_now = te_current_pct(state)
            log_progress(scenario_id, etf, "tilt_timeout", t_etf, {"iters": iters, "moves": moves, "te_calls": te_calls, "pct_clean": round(pc,2), "pct_contro": round(px,2), "te_pct": (None if np.isnan(te_now) else round(te_now,3))})
            break
    s = w.sum()
    if s>0: w = w/s
    return w, {"tilt_iters": iters, "tilt_moves": moves, "tilt_te_calls": te_calls}

def strict_exclusion(df):
    w0 = df["Weight"].values/100.0
    w = pd.Series(w0.copy(), index=df.index)
    excl = (df["FossilFuel"]+df["Tobacco"]+df["Weapons"]+df["Prisons"]+df["Deforestation"])>0
    w.loc[excl.index[excl]] = 0.0
    if abs(w.sum())<1e-12: return w
    cap = pd.Series(np.minimum(CAP_ABS, CAP_X_BASE*(w0+1e-12)), index=df.index)
    allow = ~excl
    sector_t = (df.assign(w=w)["w"].groupby(df["Sector"]).sum()/w.sum()).to_dict()
    region_t = (df.assign(w=w)["w"].groupby(df["Region"]).sum()/w.sum()).to_dict()
    w = redistribute_within(df, w, allow & (w<cap), cap, sector_t, region_t)
    s = w.sum()
    if s>0: w = w/s
    return w

def factor_drift(df, w0, w1):
    s0 = (df.assign(w=w0).groupby("Sector")["w"].sum())
    s1 = (df.assign(w=w1).groupby("Sector")["w"].sum())
    r0 = (df.assign(w=w0).groupby("Region")["w"].sum())
    r1 = (df.assign(w=w1).groupby("Region")["w"].sum())
    s0, s1 = s0/s0.sum(), s1/s1.sum()
    r0, r1 = r0/r0.sum(), r1/r1.sum()
    sd = float(np.abs(s1.reindex(s0.index, fill_value=0)-s0).sum())*100.0
    rd = float(np.abs(r1.reindex(r0.index, fill_value=0)-r0).sum())*100.0
    return sd, rd

def metrics_row(scenario_id, scope, etf, df, w_base, w_scn, cov):
    tks = df["Ticker"]
    pc_b, px_b, po_b = comp_split(df, w_base)
    pc_s, px_s, po_s = comp_split(df, w_scn)
    as_ = active_share(w_base, w_scn)
    te = te_annual(w_base, w_scn, cov, tks)
    sd, rd = factor_drift(df, w_base, w_scn)
    return {
        "scenario_id": scenario_id,
        "scope": scope,
        "etf": etf,
        "pct_clean_base": pc_b,
        "pct_contro_base": px_b,
        "pct_other_base": po_b,
        "pct_clean_scn": pc_s,
        "pct_contro_scn": px_s,
        "pct_other_scn": po_s,
        "active_share_pct": as_,
        "est_te_annual_pct": te,
        "factor_drift_sector_pct": sd,
        "factor_drift_region_pct": rd,
        "liquidity_status": "PASS",
        "notes": ""
    }

def position_deltas(scenario_id, etf, df, w_base, w_scn):
    wb = (w_base*100).round(6)
    ws = (w_scn*100).round(6)
    d = (ws - wb)
    keep = np.abs(d)>=0.05
    if not keep.any(): return pd.DataFrame(columns=["scenario_id","etf","Ticker","Name","classification","Sector","Region","weight_base_pct","weight_scn_pct","delta_weight_pct","action"])
    cls = np.where(df["Clean"]==1, "clean", np.where(df["Controversial"]==1, "controversial","other"))
    act = np.where((wb==0)&(ws>0),"ADD", np.where((wb>0)&(ws==0),"DROP", np.where(d>0,"UP","DOWN")))
    out = pd.DataFrame({
        "scenario_id": scenario_id,
        "etf": etf,
        "Ticker": df["Ticker"],
        "Name": df["Name"],
        "classification": cls,
        "Sector": df["Sector"],
        "Region": df["Region"],
        "weight_base_pct": wb,
        "weight_scn_pct": ws,
        "delta_weight_pct": d,
        "action": act
    })
    return out.loc[keep].copy()

def run():
    os.makedirs(OUT_DIR, exist_ok=True)
    t_all = perf_counter()
    u = load_universe(UNIVERSE_PATH)
    cov = load_cov(COV_PATH)
    ports = []
    poss = []
    ts = datetime.now().isoformat()
    for scenario in SCENARIOS:
        t_scn = perf_counter()
        scenario_id = scenario+"_"+datetime.now().strftime("%Y%m%dT%H%M%S")
        log_progress(scenario_id, None, "scenario_start", t_scn)
        for etf, g in u.groupby("ETF_Ticker"):
            t_etf = perf_counter()
            log_progress(scenario_id, etf, "etf_start", t_etf, {"rows": len(g)})
            df = g.reset_index(drop=True).copy()
            w0 = pd.Series(df["Weight"].values/100.0, index=df.index)
            if scenario=="Baseline":
                w1 = w0.copy()
                diag = {}
            elif scenario=="PragmaticTilt":
                w1, diag = pragmatic_tilt(scenario_id, etf, df, cov, TE_BUDGET_TILT, t_etf)
                s = w1.sum()
                if s>0: w1 = w1/s
            elif scenario=="StrictExclusion":
                w1 = strict_exclusion(df)
                diag = {}
                s = w1.sum()
                if s>0: w1 = w1/s
            else:
                w1 = w0.copy()
                diag = {}
            log_progress(scenario_id, etf, "weights_ready", t_etf, diag)
            ports.append(metrics_row(scenario_id, "etf", etf, df, w0, w1, cov))
            poss_df = position_deltas(scenario_id, etf, df, w0, w1)
            if not poss_df.empty:
                poss.append(poss_df)
            log_progress(scenario_id, etf, "positions_done", t_etf, {"pos_rows": 0 if poss_df.empty else len(poss_df)})
        log_progress(scenario_id, None, "scenario_complete", t_scn)
    ports_df = pd.DataFrame(ports)
    agg_rows = []
    for sid, grp in ports_df.groupby("scenario_id"):
        agg = {"scenario_id": sid, "scope":"aggregate", "etf": None}
        for k in ["pct_clean_base","pct_contro_base","pct_other_base","pct_clean_scn","pct_contro_scn","pct_other_scn","active_share_pct","est_te_annual_pct","factor_drift_sector_pct","factor_drift_region_pct"]:
            agg[k] = grp[k].mean(skipna=True)
        agg["liquidity_status"] = "PASS"
        agg["notes"] = ""
        agg_rows.append(agg)
    ports_df = pd.concat([ports_df, pd.DataFrame(agg_rows)], ignore_index=True)
    ports_df["rule_te_budget_annual_pct"] = TE_BUDGET_TILT
    ports_df["rule_sector_band"] = SECTOR_BAND
    ports_df["rule_region_band"] = REGION_BAND
    ports_df["rule_cap_abs_pct"] = CAP_ABS
    ports_df["rule_cap_x_baseline"] = CAP_X_BASE
    ports_df["rule_clean_tilt_target_pp"] = np.where(ports_df["scenario_id"].str.startswith("PragmaticTilt_"), CLEAN_TILT_TARGET_PP, np.nan)
    is_strict = ports_df["scenario_id"].str.startswith("StrictExclusion_")
    ports_df["rule_excl_fossil"] = is_strict
    ports_df["rule_excl_tobacco"] = is_strict
    ports_df["rule_excl_weapons"] = is_strict
    ports_df["rule_excl_prisons"] = is_strict
    ports_df["rule_excl_deforestation"] = is_strict
    pos_df = pd.concat(poss, ignore_index=True) if len(poss)>0 else pd.DataFrame(columns=["scenario_id","etf","Ticker","Name","classification","Sector","Region","weight_base_pct","weight_scn_pct","delta_weight_pct","action"])
    ports_df.to_csv(PORT_PATH, index=False)
    pos_df.to_csv(POS_PATH, index=False)
    log_progress("ALL", None, "all_complete", t_all, {"ports_rows": len(ports_df), "pos_rows": len(pos_df)})
    print(f"Wrote {PORT_PATH} rows={len(ports_df)}")
    print(f"Wrote {POS_PATH} rows={len(pos_df)}")

if __name__=="__main__":
    run()


[2025-10-17T17:07:42.851046] Baseline_20251017T170742 |  | scenario_start | 0.0s
[2025-10-17T17:07:42.855964] Baseline_20251017T170742 | CRBN | etf_start | 0.0s {'rows': 1076}
[2025-10-17T17:07:42.857137] Baseline_20251017T170742 | CRBN | weights_ready | 0.0012s {}
[2025-10-17T17:07:42.877790] Baseline_20251017T170742 | CRBN | positions_done | 0.0218s {'pos_rows': 0}
[2025-10-17T17:07:42.881790] Baseline_20251017T170742 | DMXF | etf_start | 0.0s {'rows': 472}
[2025-10-17T17:07:42.883301] Baseline_20251017T170742 | DMXF | weights_ready | 0.0015s {}
[2025-10-17T17:07:42.892550] Baseline_20251017T170742 | DMXF | positions_done | 0.0108s {'pos_rows': 0}
[2025-10-17T17:07:42.893363] Baseline_20251017T170742 | DSI | etf_start | 0.0s {'rows': 426}
[2025-10-17T17:07:42.894135] Baseline_20251017T170742 | DSI | weights_ready | 0.0008s {}
[2025-10-17T17:07:42.898755] Baseline_20251017T170742 | DSI | positions_done | 0.0054s {'pos_rows': 0}
[2025-10-17T17:07:42.899422] Baseline_20251017T170742 | E

In [45]:
#Analysis 3 
import os, re, time
import pandas as pd
import numpy as np

# ---------- Paths & knobs ----------
BASE_DIR   = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data"
PRICES_DIR = os.path.join(BASE_DIR, "Prices")
OUT_DIR    = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 3"
os.makedirs(OUT_DIR, exist_ok=True)

HOLDINGS_2025_CSV = os.path.join(BASE_DIR, "holdings_2025_final.csv")
PRICES_XLSX       = os.path.join(PRICES_DIR, "holdings_2025_prices_final.xlsx")
CLASS_BIN_CSV     = os.path.join(BASE_DIR, "classification_binary.csv")

UNIVERSE_XLSX     = os.path.join(OUT_DIR, "universe_2025.xlsx")
RETURNS_CSV       = os.path.join(OUT_DIR, "returns_top_per_etf_2025.csv")
SYMBOL_MAP_CSV    = os.path.join(OUT_DIR, "yahoo_symbol_map.csv")
COV_OUT           = os.path.join(OUT_DIR, "covariance_2025.csv")

TOP_PER_ETF     = 20           # or set COVERAGE_FLOOR for cumulative coverage
COVERAGE_FLOOR  = None         # e.g., 0.8 for 80% coverage (overrides TOP_PER_ETF if set)
PERIOD          = "6mo"
BATCH_SIZE      = 80
SLEEP_SEC       = 1.0
WINSOR_LO_HI    = (0.01, 0.99)
MIN_RET_OBS     = 60           # drop tickers with fewer daily return obs (stability)

# ---------- Helpers ----------
def pick_col(cols, names):
    low = {c.lower(): c for c in cols}
    for n in names:
        key = n.lower()
        if key in low:
            return low[key]
    def norm(s): return re.sub(r"[^a-z0-9]", "", s.lower())
    norm_map = {norm(c): c for c in cols}
    for n in names:
        nn = norm(n)
        if nn in norm_map:
            return norm_map[nn]
    for c in cols:
        cc = re.sub(r"[^a-z0-9]", "", c.lower())
        for n in names:
            if "|" in n:
                parts = [re.sub(r"[^a-z0-9]", "", x.lower()) for x in n.split("|")]
                if all(p in cc for p in parts):
                    return c
    return None

def norm_name(s):
    if pd.isna(s): return None
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def clean_weight(series):
    s = pd.to_numeric(series, errors="coerce")
    if s.dropna().between(0,1).mean() > 0.8 and s.max() <= 1.5:
        s = s * 100.0
    return s

SUFFIX_MAP = {
    "united states":"", "usa":"", "us":"", "u.s.":"", "u.s":"",
    "canada":".TO", "brazil":".SA",
    "united kingdom":".L", "uk":".L", "great britain":".L", "england":".L",
    "germany":".DE", "france":".PA", "netherlands":".AS", "belgium":".BR",
    "italy":".MI", "spain":".MC", "portugal":".LS", "sweden":".ST",
    "denmark":".CO", "norway":".OL", "finland":".HE", "switzerland":".SW",
    "austria":".VI", "ireland":".IR",
    "japan":".T", "hong kong":".HK", "china":".SS", "taiwan":".TW",
    "south korea":".KS", "korea":".KS", "india":".NS", "australia":".AX",
    "singapore":".SI", "new zealand":".NZ",
    "europe":".PA", "emea":".PA", "asia":".T", "apac":".T"
}

MANUAL_MAP = {
    "BRK.B":"BRK-B", "BF.B":"BF-B",
    "NOVO B":"NOVO-B.CO", "NOVOB":"NOVO-B.CO", "NOVO-B":"NOVO-B.CO",
    "ORSTED":"ORSTED.CO", "VWS":"VWS.CO",
    "NOVN":"NOVN.SW", "NESN":"NESN.SW", "UBSG":"UBSG.SW", "ZURN":"ZURN.SW", "SREN":"SREN.SW", "GIVN":"GIVN.SW",
    "HSBA":"HSBA.L", "LLOY":"LLOY.L", "BARC":"BARC.L", "DGE":"DGE.L", "REL":"REL.L", "LR":"LR.PA",
    "SIE":"SIE.DE", "IFX":"IFX.DE", "DB1":"DB1.DE", "NDX1":"NDX1.DE", "LONN":"LONN.SW",
    "IBE":"IBE.MC", "EDP":"EDP.LS", "EDPR":"EDPR.LS",
    "UCG":"UCG.MI", "INGA":"INGA.AS", "PRX":"PRX.AS",
    "700":"0700.HK", "968":"0968.HK", "388":"0388.HK", "2318":"2318.HK", "2015":"2015.HK", "9988":"9988.HK",
    "3690":"3690.HK", "9868":"9868.HK", "1299":"1299.HK", "1928":"1928.HK", "1120":"1120.HK", "66":"0066.HK",
    "2330":"2330.TW",
    "RELIANCE":"RELIANCE.NS", "HDFCBANK":"HDFCBANK.NS", "HINDUNILVR":"HINDUNILVR.NS", "NHPC":"NHPC.NS", "SUZLON":"SUZLON.NS",
    "600900":"600900.SS",
    "NPI":"NPI.TO", "WSP":"WSP.TO", "MOWI":"MOWI.OL",
    "XTSLA": None, "9434-T.T":"9434.T",
    "288.HK":"0288.HK", "660":"0660.HK", "268.SS":"0268.HK", "9999.SS":"9999.HK",
    "3988.SS":"3988.HK", "2269.SS":"2269.HK",
    "IDCBY.SS":"IDCBY", "SPOT.ST":"SPOT", "ESSITYB.ST":"ESSITY-B.ST", "373220":"373220.KS",
    "NPN":"NPN.JO"
}

def normalize_tkr(t):
    if t is None or (isinstance(t, float) and pd.isna(t)): return None
    return str(t).strip().upper().replace("  ", " ")

def guess_yahoo_symbol(ticker, location):
    t0 = normalize_tkr(ticker)
    if not t0: return None, "empty_ticker"
    if t0 in MANUAL_MAP: return MANUAL_MAP[t0], "manual_map"
    t_cls = t0.replace(".", "-").replace(" ", "")
    loc = (location or "").strip().lower()
    suffix = SUFFIX_MAP.get(loc, "")
    return f"{t_cls}{suffix}", f"suffix_by_location({loc or 'unknown'})"

def winsorize_grouped(prices, lo=0.01, hi=0.99):
    def _clip(s):
        a, b = s.quantile(lo), s.quantile(hi)
        return s.clip(a, b)
    prices["ret"] = prices.groupby("ticker_yahoo", dropna=False)["ret"].transform(_clip)
    return prices

# ---------- Phase 1: Build Universe ----------
def phase1_build_universe():
    h = pd.read_csv(HOLDINGS_2025_CSV, dtype=str)
    for c in h.columns:
        if h[c].dtype == object:
            h[c] = h[c].astype(str).str.strip()

    etf_col = pick_col(h.columns, ["ETF_Ticker","ETF Ticker","etf_ticker","fund","etf"])
    tic_col = pick_col(h.columns, ["company_ticker","ticker","symbol"])
    nn_col  = pick_col(h.columns, ["name_normalised","name_normalized","company_name","name"])
    sec_col = pick_col(h.columns, ["Sector"])
    loc_col = pick_col(h.columns, ["Location","Country","Region"])
    w_col   = pick_col(h.columns, ["Weight (%)","weight (%)","Weight%","weight","portfolio_weight_pct"])
    mv_col  = pick_col(h.columns, ["Market Value","market value","market_value_usd","market_value"])

    for req_col, label in [(etf_col,"ETF_Ticker"),(tic_col,"company_ticker"),
                           (nn_col,"name_normalized"),(sec_col,"Sector"),
                           (loc_col,"Location/Region"),(w_col,"Weight (%)"),
                           (mv_col,"Market Value")]:
        if req_col is None:
            raise ValueError(f"Required column missing for: {label}")

    base = h[[etf_col, tic_col, nn_col, sec_col, loc_col, w_col, mv_col]].copy()
    base.columns = ["ETF_Ticker","company_ticker","name_normalized","Sector","Location","Weight (%)","Market Value"]
    base["_row_order"] = np.arange(len(base))
    base["_name_key"] = base["name_normalized"].map(norm_name)

    p = pd.read_excel(PRICES_XLSX, dtype=str)
    for c in p.columns:
        if p[c].dtype == object:
            p[c] = p[c].astype(str).str.strip()
    p_name  = pick_col(p.columns, ["company_name","name","name_normalized","name_normalised"])
    p_price = pick_col(p.columns, ["price","price_usd","usd_price","price usd"])
    if p_name is None or p_price is None:
        raise ValueError("Missing price columns in prices file.")
    p_map = p[[p_name, p_price]].drop_duplicates().rename(columns={p_name:"company_name", p_price:"price"})
    p_map["_name_key"] = p_map["company_name"].map(norm_name)

    m_prices = base.merge(p_map[["_name_key","price"]], how="left", on="_name_key")
    if m_prices["price"].isna().any():
        p_tic = pick_col(p.columns, ["ticker","symbol","company_ticker"])
        if p_tic:
            p_map2 = p[[p_tic, p_price]].drop_duplicates().rename(
                columns={p_tic:"company_ticker", p_price:"price_by_ticker"}
            )
            m_prices = m_prices.merge(p_map2, how="left", on="company_ticker")
            m_prices["price"] = m_prices["price"].fillna(m_prices["price_by_ticker"])
            m_prices = m_prices.drop(columns=["price_by_ticker"], errors="ignore")

    cb = pd.read_csv(CLASS_BIN_CSV, dtype=str)
    for c in cb.columns:
        if cb[c].dtype == object:
            cb[c] = cb[c].astype(str).str.strip()
    cb_etf  = pick_col(cb.columns, ["ETF Ticker","ETF_Ticker","etf_ticker","fund","etf"])
    cb_tic  = pick_col(cb.columns, ["company_ticker","ticker","symbol"])
    cb_name = pick_col(cb.columns, ["name_normalized","name_normalised","name","company_name"])
    cb_c200 = pick_col(cb.columns, ["clean200","clean_200"])
    cb_def  = pick_col(cb.columns, ["deforestation"])
    cb_fos  = pick_col(cb.columns, ["fossil fuel","fossil fuels","fossilfuel","fossil_fuel"])
    cb_pri  = pick_col(cb.columns, ["prison","prisons","private prisons","privateprisons","private_prisons"])
    cb_tob  = pick_col(cb.columns, ["tobacco"])
    cb_wep  = pick_col(cb.columns, ["weapons","weapon","controversial weapons","controversialweapons","controversial_weapons"])
    if cb_name is None:
        raise ValueError("classification.name column missing")

    flag_cols  = [cb_c200, cb_fos, cb_tob, cb_wep, cb_pri, cb_def]
    flag_names = ["Clean200","FossilFuel","Tobacco","Weapons","Prisons","Deforestation"]

    cb_flags = cb[[c for c in [cb_etf, cb_tic, cb_name] if c is not None] + [c for c in flag_cols if c is not None]].copy()
    rename_map = {}
    if cb_etf:  rename_map[cb_etf] = "ETF_Ticker"
    if cb_tic:  rename_map[cb_tic] = "company_ticker"
    rename_map[cb_name] = "name_normalized"
    cb_flags = cb_flags.rename(columns=rename_map)
    cb_flags["_name_key"] = cb_flags["name_normalized"].map(norm_name)

    for src, tgt in zip(flag_cols, flag_names):
        if src and src in cb_flags.columns:
            cb_flags[tgt] = cb_flags[src]
        if tgt not in cb_flags.columns:
            cb_flags[tgt] = 0
        cb_flags[tgt] = cb_flags[tgt].map({"1":1,"0":0,"Y":1,"N":0,"Yes":1,"No":0,1:1,0:0}).fillna(0).astype(int)

    group_keys = [k for k in ["ETF_Ticker","company_ticker","name_normalized"] if k in cb_flags.columns]
    cb_agg = cb_flags.groupby(group_keys, dropna=False)[flag_names].max().reset_index()
    cb_agg["_name_key"] = cb_agg["name_normalized"].map(norm_name)

    m_cls = m_prices.copy()
    if all(k in m_cls.columns for k in ["ETF_Ticker","company_ticker","name_normalized"]) and \
       all(k in cb_agg.columns for k in ["ETF_Ticker","company_ticker","name_normalized"]):
        m_cls = m_cls.merge(
            cb_agg[["ETF_Ticker","company_ticker","name_normalized"] + flag_names],
            how="left",
            on=["ETF_Ticker","company_ticker","name_normalized"],
        )

    missing_mask = m_cls[flag_names].isna().all(axis=1)
    if missing_mask.any():
        fill = m_cls.loc[missing_mask, ["_name_key"]].merge(
            cb_agg[["_name_key"] + flag_names].drop_duplicates("_name_key"),
            how="left",
            on="_name_key"
        )
        for col in flag_names:
            if col in fill.columns:
                m_cls.loc[missing_mask, col] = fill[col].values
            else:
                m_cls.loc[missing_mask, col] = 0

    for col in flag_names:
        if col not in m_cls.columns:
            m_cls[col] = 0
        m_cls[col] = m_cls[col].fillna(0).astype(int)

    # Resolver: if clean AND controversial -> controversial wins
    m_cls["Controversies"] = ((m_cls["FossilFuel"] + m_cls["Tobacco"] + m_cls["Weapons"] + m_cls["Prisons"] + m_cls["Deforestation"]) > 0).astype(int)
    m_cls["Clean_raw"] = (m_cls["Clean200"] > 0).astype(int)
    m_cls["Clean"] = np.where(m_cls["Controversies"] == 1, 0, m_cls["Clean_raw"]).astype(int)
    m_cls["Resolver"] = np.where(
        m_cls["Controversies"] == 1, "Controversial",
        np.where(m_cls["Clean200"] == 1, "Clean", "Other")
    )

    m_cls = m_cls.sort_values("_row_order")
    if "company_name" not in m_cls.columns:
        m_cls["company_name"] = m_cls["name_normalized"]

    out_cols = [
        "ETF_Ticker","company_ticker","company_name","name_normalized",
        "Sector","Location","Weight (%)","Market Value","price",
        "Clean200","FossilFuel","Tobacco","Weapons","Prisons","Deforestation",
        "Controversies","Clean","Resolver"
    ]
    m_cls[out_cols].to_excel(UNIVERSE_XLSX, index=False)
    print(f"[phase1] Wrote {UNIVERSE_XLSX}  rows={len(m_cls):,}")

# ---------- Selection (top per ETF or coverage floor) ----------
def select_top_per_etf(uni, c_etf, c_tic, c_w, c_loc):
    df = uni.copy()
    df["_etf"] = df[c_etf].astype(str).str.strip().str.upper()
    df["_tic"] = df[c_tic].astype(str).str.strip().str.upper()
    df["_w"] = clean_weight(df[c_w])
    if c_loc:
        df["_loc"] = df[c_loc].astype(str)
    else:
        df["_loc"] = pd.Series(["United States"] * len(df), index=df.index)
    selected = []
    for etf, g in df.groupby("_etf", sort=False):
        g2 = g.sort_values("_w", ascending=False).reset_index(drop=True)
        if COVERAGE_FLOOR:
            g2["_cum"] = g2["_w"].fillna(0).cumsum() / g2["_w"].fillna(0).sum()
            keep = g2[g2["_cum"] <= float(COVERAGE_FLOOR)]
        else:
            keep = g2.head(int(TOP_PER_ETF))
        selected.append(keep[["_etf","_tic","_w","_loc"]])
    sel = pd.concat(selected, ignore_index=True)
    sel["_key"] = sel["_tic"]
    agg = sel.sort_values("_w", ascending=False).drop_duplicates("_key")
    etfs_by_ticker = sel.groupby("_tic", dropna=False)["_etf"].apply(lambda s: ";".join(sorted(set(s)))).reset_index().rename(columns={"_tic":"ticker","_etf":"etfs"})
    agg = agg.rename(columns={"_tic":"ticker","_w":"agg_weight_pct","_loc":"location"}).drop(columns=["_key"])
    out = agg.merge(etfs_by_ticker, on="ticker", how="left")
    return out

# ---------- Phase 2: Returns ----------
def phase2_build_returns():
    import yfinance as yf
    from tqdm import tqdm

    if not os.path.exists(UNIVERSE_XLSX):
        raise FileNotFoundError("universe_2025.xlsx not found (run Phase 1).")
    uni = pd.read_excel(UNIVERSE_XLSX)

    cols = list(uni.columns)
    c_etf = pick_col(cols, ["ETF_Ticker","etf","fund"])
    c_tic = pick_col(cols, ["company_ticker","ticker","symbol"])
    c_w   = pick_col(cols, ["Weight (%)","weight (%)","Weight%","weight","portfolio_weight_pct"])
    c_loc = pick_col(cols, ["Location","Country","Region","location"])
    if not (c_etf and c_tic and c_w):
        raise ValueError("universe must have ETF, ticker, and weight columns.")

    top = select_top_per_etf(uni, c_etf, c_tic, c_w, c_loc)

    # Map to Yahoo symbols
    rows = []
    for _, r in top.iterrows():
        sym, note = guess_yahoo_symbol(r["ticker"], r.get("location"))
        rows.append({**r.to_dict(), "yahoo_symbol": sym, "notes": note})
    ymap = pd.DataFrame(rows)
    ymap["succeeded"] = False
    ymap.to_csv(SYMBOL_MAP_CSV, index=False)

    symbols = [s for s in ymap["yahoo_symbol"].dropna().tolist() if s]

    frames = []
    ok_syms = set()
    for i in tqdm(range(0, len(symbols), BATCH_SIZE), desc="Downloading"):
        batch = symbols[i:i+BATCH_SIZE]
        try:
            data = yf.download(batch, period=PERIOD, interval="1d", auto_adjust=True, progress=False, group_by="ticker", threads=True)
        except Exception:
            time.sleep(SLEEP_SEC)
            continue
        if data is None or data.empty:
            time.sleep(SLEEP_SEC)
            continue

        if isinstance(data.columns, pd.MultiIndex):
            for sym in batch:
                try:
                    sub = data[sym]
                except KeyError:
                    continue
                pick = None
                for cname in sub.columns:
                    c0 = cname.lower().replace(" ", "")
                    if c0 in ("adjclose","close","price"):
                        pick = cname
                        break
                if pick is None:
                    continue
                px = sub[pick].dropna()
                if px.empty:
                    continue
                dfp = px.to_frame("px").reset_index()
                dfp.rename(columns={dfp.columns[0]:"date"}, inplace=True)
                dfp["ticker_yahoo"] = sym
                ok_syms.add(sym)
                frames.append(dfp[["date","ticker_yahoo","px"]])
        else:
            cols_lower = [c.lower().replace(" ","") for c in data.columns]
            pick = None
            for cname, c0 in zip(data.columns, cols_lower):
                if c0 in ("adjclose","close","price"):
                    pick = cname
                    break
            if pick is not None:
                px = data[pick].dropna()
                if not px.empty:
                    dfp = px.to_frame("px").reset_index()
                    dfp.rename(columns={dfp.columns[0]:"date"}, inplace=True)
                    dfp["ticker_yahoo"] = batch[0]
                    ok_syms.add(batch[0])
                    frames.append(dfp[["date","ticker_yahoo","px"]])
        time.sleep(SLEEP_SEC)

    if not frames:
        raise RuntimeError("No price data downloaded.")

    # Update mapping successes
    ymap["succeeded"] = ymap["yahoo_symbol"].isin(ok_syms)
    ymap.to_csv(SYMBOL_MAP_CSV, index=False)

    prices = pd.concat(frames, ignore_index=True)
    prices["date"] = pd.to_datetime(prices["date"], errors="coerce")
    prices = prices.dropna(subset=["date"]).sort_values(["ticker_yahoo","date"]).reset_index(drop=True)

    prices["ret"] = prices.groupby("ticker_yahoo", dropna=False)["px"].pct_change()
    prices = prices.dropna(subset=["ret"]).reset_index(drop=True)

    # Optional winsorization
    if WINSOR_LO_HI:
        lo, hi = WINSOR_LO_HI
        prices = winsorize_grouped(prices, lo=lo, hi=hi)

    # Filter short histories
    counts = prices.groupby("ticker_yahoo")["ret"].count()
    keep = counts[counts >= MIN_RET_OBS].index
    prices = prices[prices["ticker_yahoo"].isin(keep)].copy()

    rev = dict(zip(ymap["yahoo_symbol"], ymap["ticker"]))
    prices["ticker"] = prices["ticker_yahoo"].map(rev)
    prices = prices.dropna(subset=["ticker"])

    meta = ymap.set_index("ticker")[["etfs","location","agg_weight_pct","yahoo_symbol"]]
    out = prices.merge(meta, on="ticker", how="left")[["date","ticker","ret","etfs","location","agg_weight_pct","yahoo_symbol"]]
    out = out.sort_values(["ticker","date"]).reset_index(drop=True)
    out.to_csv(RETURNS_CSV, index=False)
    print(f"[phase2] Wrote {RETURNS_CSV}  rows={len(out):,}  tickers={out['ticker'].nunique()}")

# ---------- Phase 3: Covariance ----------
def phase3_build_cov():
    if not os.path.exists(RETURNS_CSV):
        raise FileNotFoundError("returns_top_per_etf_2025.csv not found (run Phase 2).")
    r = pd.read_csv(RETURNS_CSV)
    r["date"] = pd.to_datetime(r["date"], errors="coerce")
    r = r.dropna(subset=["date"])

    r.columns = [c.lower() for c in r.columns]
    req = {"date", "ticker", "ret"}
    if not req.issubset(set(r.columns)):
        missing = req - set(r.columns)
        raise ValueError(f"Missing required columns in returns file: {missing}")

    r_pivot = r.pivot(index="date", columns="ticker", values="ret").fillna(0.0)
    cov = r_pivot.cov()
    if isinstance(cov, pd.Series):
        cov = cov.to_frame()
    cov.to_csv(COV_OUT, float_format="%.10f")
    print(f"[phase3] Wrote {COV_OUT}  shape={cov.shape}")

if __name__ == "__main__":
    phase1_build_universe()
    phase2_build_returns()
    phase3_build_cov()


[phase1] Wrote /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 3/universe_2025.xlsx  rows=8,249


Downloading:   0%|                                        | 0/3 [00:00<?, ?it/s]
7 Failed downloads:
['ESSITYB.ST', '288.HK', 'BRKB', '5930', '939.SS', '268.SS', '1810.SS']: YFPricesMissingError('possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")')
Downloading:  33%|██████████▋                     | 1/3 [00:08<00:16,  8.05s/it]
8 Failed downloads:
['3988.SS', '1398.SS', 'SPOT.ST', '9618.SS', '9961.SS', '9999.SS', 'NU.SA', '316140']: YFPricesMissingError('possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")')
Downloading:  67%|█████████████████████▎          | 2/3 [00:17<00:08,  8.63s/it]
1 Failed download:
['105560']: YFPricesMissingError('possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")')
Downloading: 100%|████████████████████████████████| 3/3 [00:20<00:00,  6.99s/it]


[phase2] Wrote /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 3/returns_top_per_etf_2025.csv  rows=22,333  tickers=176
[phase3] Wrote /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 3/covariance_2025.csv  shape=(176, 176)


In [46]:
# ===== Analysis 3 — Code 2: Run portfolio scenarios (CSV-only outputs) =====
# Inputs:
#   - universe_2025.xlsx (or universe_2025.csv)
#   - covariance_2025.csv (optional but recommended)
# Outputs:
#   - scenario_specs.csv
#   - scenario_portfolio_metrics.csv
#   - scenario_position_deltas.csv
#   - scenario_progress.csv

import os, math, time, re
import pandas as pd
import numpy as np

# ---------- Paths ----------
BASE  = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data"
DASH3 = os.path.join(BASE, "Data for Dashboard", "Analysis 3")
os.makedirs(DASH3, exist_ok=True)

UNIVERSE_XLSX = os.path.join(DASH3, "universe_2025.xlsx")
UNIVERSE_CSV  = os.path.join(DASH3, "universe_2025.csv")   # optional fallback
COV_CSV       = os.path.join(DASH3, "covariance_2025.csv")

SPECS_CSV   = os.path.join(DASH3, "scenario_specs.csv")
METRICS_CSV = os.path.join(DASH3, "scenario_portfolio_metrics.csv")
DELTAS_CSV  = os.path.join(DASH3, "scenario_position_deltas.csv")
PROG_CSV    = os.path.join(DASH3, "scenario_progress.csv")

# ---------- Scenarios ----------
SCENARIOS = [
    {
        "name":"Baseline",
        "type":"baseline",
        "te_budget_annual": None,
        "sector_neutrality_pct": 0.0,   # tolerance band ±0% (exact match)
        "region_neutrality_pct": 0.0,   # tolerance band ±0%
        "single_name_cap_pct": 5.0,
        "single_name_cap_mult": 3.0,
        "hard_screens": []
    },
    {
        "name":"Pragmatic Tilt",
        "type":"tilt",
        "te_budget_annual": 0.02,       # 2% annual TE budget (approx)
        "sector_neutrality_pct": 2.0,   # tolerance band ±2%
        "region_neutrality_pct": 2.0,
        "single_name_cap_pct": 5.0,
        "single_name_cap_mult": 3.0,
        "hard_screens": []
    },
    {
        "name":"Strict Exclusion",
        "type":"exclude",
        "te_budget_annual": None,       # free TE; hard screen applies
        "sector_neutrality_pct": 2.0,
        "region_neutrality_pct": 2.0,
        "single_name_cap_pct": 5.0,
        "single_name_cap_mult": 3.0,
        "hard_screens": ["Controversial"]
    }
]

# ---------- Helpers ----------
def pick_col(cols, names):
    low = {c.lower(): c for c in cols}
    for n in names:
        if n.lower() in low:
            return low[n.lower()]
    def simp(s): return re.sub(r"[^a-z0-9]", "", s.lower())
    sm = {simp(c): c for c in cols}
    for n in names:
        nn = simp(n)
        if nn in sm:
            return sm[nn]
    return None

def norm_pct(s):
    x = pd.to_numeric(s, errors="coerce")
    if x.dropna().between(0,1).mean() > 0.8 and x.max() <= 1.5: x = x*100
    return x

def cap_vector(baseline_w, new_w, cap_pct=5.0, cap_mult=3.0):
    # Cap at min(absolute cap, multiple of baseline)
    cap_abs = cap_pct/100.0
    cap_rel = np.maximum(0.0, baseline_w)*cap_mult
    return np.minimum(new_w, np.minimum(cap_abs, cap_rel))

def renorm_to(target_sum, w):
    s = float(np.nansum(w))
    if s <= 0: return w
    return w * (target_sum / s)

# --- Region bucketing (simple) ---
def to_region(loc_str):
    if not isinstance(loc_str, str): return "Other"
    s = loc_str.strip().lower()
    if any(k in s for k in ["united states","u.s.","us","usa"]): return "US"
    if any(k in s for k in ["canada","mexico","brazil","chile","argentina","colombia"]): return "Americas"
    if any(k in s for k in ["uk","united kingdom","france","germany","spain","italy","europe","netherlands","belgium","switzerland","sweden","denmark","norway","finland","austria","portugal","ireland"]): return "Europe"
    if any(k in s for k in ["japan","china","hong kong","taiwan","south korea","india","singapore","australia","new zealand","asia","apac"]): return "APAC"
    return "Other"

# --- Neutrality with tolerance bands (vectorized; no groupby.apply warnings) ---
def refill_with_tolerance(df, group_col, base_sum_map, band_pct):
    """
    If deviation of new weights from baseline sum exceeds the tolerance band (±band_pct),
    snap the group's new sum back to the baseline sum. Otherwise leave unchanged.
    """
    df = df.copy()
    # current sums (new) and baseline
    cur = df.groupby(group_col, dropna=False)["w_new"].sum()
    base= df.groupby(group_col, dropna=False)["w_base"].sum()

    # Build scaling factors
    factors = {}
    for g in sorted(set(cur.index) | set(base.index)):
        base_sum = float(base_sum_map.get(g, base.get(g, 0.0)))
        cur_sum  = float(cur.get(g, 0.0))
        if base_sum <= 0.0:
            # nothing to match; leave as is
            factors[g] = 1.0
            continue
        # deviation as % of base
        dev = (cur_sum - base_sum) / base_sum if base_sum != 0 else 0.0
        if abs(dev) > (band_pct/100.0):
            # snap back exactly to baseline
            factors[g] = (base_sum / cur_sum) if cur_sum != 0 else 1.0
        else:
            # within tolerance: no change
            factors[g] = 1.0

    df["w_new"] = df["w_new"] * df[group_col].map(factors).fillna(1.0)
    return df

def compute_resolver_flags(df):
    if "Clean" not in df.columns and "Resolver" in df.columns:
        df["Clean"] = (df["Resolver"].astype(str).str.lower() == "clean").astype(int)
    if "Controversies" not in df.columns and "Resolver" in df.columns:
        df["Controversies"] = (df["Resolver"].astype(str).str.lower() == "controversial").astype(int)
    return df

def to_port_vec(df, ticker_col, w_col):
    return df.groupby(ticker_col, dropna=False)[w_col].sum()

def tracking_error_annual(b_w, s_w, cov):
    tick = sorted(set(b_w.index) & set(s_w.index) & set(cov.index) & set(cov.columns))
    if not tick: return np.nan
    d = (s_w.reindex(tick).fillna(0.0) - b_w.reindex(tick).fillna(0.0)).values
    C = cov.reindex(tick).reindex(columns=tick).values
    te_daily_var = float(np.dot(d, C.dot(d)))
    if te_daily_var < 0: te_daily_var = 0.0
    te_daily = math.sqrt(te_daily_var)
    return te_daily * math.sqrt(252.0)

def scenario_apply_baseline(g):
    g = g.copy()
    g["w_new"] = g["w_base"].values
    return g

def scenario_apply_exclude(g, hard_screens, cap_pct, cap_mult, base_sector_sums, base_region_sums, sec_band, reg_band):
    g = g.copy()
    # hard screen
    if "Controversial" in hard_screens:
        g["w_new"] = np.where(g["is_contro"]==1, 0.0, g["w_base"])
    else:
        g["w_new"] = g["w_base"].values

    # caps
    g["w_new"] = cap_vector(g["w_base"].values, g["w_new"].values, cap_pct, cap_mult)

    # neutrality with tolerance bands
    if "Sector" in g.columns:
        g = refill_with_tolerance(g, "Sector", base_sector_sums, sec_band)
    if "Region" in g.columns:
        g = refill_with_tolerance(g, "Region", base_region_sums, reg_band)

    # final renorm to ETF total (sum of baseline)
    g["w_new"] = renorm_to(float(g["w_base"].sum()), g["w_new"].values)
    return g

def scenario_apply_tilt(g, alpha, cap_pct, cap_mult, base_sector_sums, base_region_sums, sec_band, reg_band):
    g = g.copy()
    # proportional reallocation from controversial to non-contro
    good = (g["is_contro"]==0)
    bad  = (g["is_contro"]==1)
    w = g["w_base"].values.copy()
    shift_pool = alpha * float(g.loc[bad, "w_base"].sum())
    if float(g.loc[bad, "w_base"].sum()) > 0 and float(g.loc[good, "w_base"].sum()) > 0:
        g["w_new"] = w.copy()
        g.loc[bad,  "w_new"] = g.loc[bad,  "w_new"] * (1 - alpha)
        g.loc[good, "w_new"] = g.loc[good, "w_new"] + (g.loc[good,"w_base"] / float(g.loc[good,"w_base"].sum())) * shift_pool
    else:
        g["w_new"] = w

    # caps
    g["w_new"] = cap_vector(g["w_base"].values, g["w_new"].values, cap_pct, cap_mult)

    # neutrality with tolerance bands
    if "Sector" in g.columns:
        g = refill_with_tolerance(g, "Sector", base_sector_sums, sec_band)
    if "Region" in g.columns:
        g = refill_with_tolerance(g, "Region", base_region_sums, reg_band)

    # final renorm
    g["w_new"] = renorm_to(float(g["w_base"].sum()), g["w_new"].values)
    return g

def summarize_portfolio(g):
    tot = float(g["w_new"].sum())
    clean = float(g.loc[g["is_clean"]==1, "w_new"].sum())
    contro= float(g.loc[g["is_contro"]==1,"w_new"].sum())
    other = max(0.0, tot - clean - contro)
    return pd.Series({
        "weight_sum": tot,
        "pct_clean": (clean/tot*100.0 if tot>0 else 0.0),
        "pct_contro": (contro/tot*100.0 if tot>0 else 0.0),
        "pct_other": (other/tot*100.0 if tot>0 else 0.0)
    })

# ---------- Main ----------
def main():
    # 1) specs
    pd.DataFrame([{
        "scenario": s["name"],
        "type": s["type"],
        "te_budget_annual": ("" if s["te_budget_annual"] is None else s["te_budget_annual"]),
        "sector_neutrality_pct": s["sector_neutrality_pct"],
        "region_neutrality_pct": s["region_neutrality_pct"],
        "single_name_cap_pct": s["single_name_cap_pct"],
        "single_name_cap_mult": s["single_name_cap_mult"],
        "hard_screens": ";".join(s["hard_screens"]) if s["hard_screens"] else ""
    } for s in SCENARIOS]).to_csv(SPECS_CSV, index=False)

    # 2) universe
    if os.path.exists(UNIVERSE_XLSX):
        u = pd.read_excel(UNIVERSE_XLSX)
    elif os.path.exists(UNIVERSE_CSV):
        u = pd.read_csv(UNIVERSE_CSV)
    else:
        raise FileNotFoundError("Universe not found: expected universe_2025.xlsx or universe_2025.csv in Analysis 3 folder")

    cols = list(u.columns)
    c_etf = pick_col(cols, ["ETF_Ticker","ETF Ticker","etf_ticker","fund","etf"])
    c_tic = pick_col(cols, ["company_ticker","ticker","symbol"])
    c_name= pick_col(cols, ["company_name","name","name_normalized","name_normalised"])
    c_sec = pick_col(cols, ["Sector"])
    c_loc = pick_col(cols, ["Location","Country","Region","location"])
    c_w   = pick_col(cols, ["Weight (%)","weight (%)","Weight%","weight","portfolio_weight_pct"])
    if not (c_etf and c_tic and c_w): raise ValueError("universe must have ETF, ticker, and weight cols")

    u = compute_resolver_flags(u.copy())
    u["w_base"] = norm_pct(u[c_w]).astype(float)/100.0
    u["Sector"] = (u[c_sec].astype(str) if c_sec else "ALL")
    u["Location"] = (u[c_loc].astype(str) if c_loc else "United States")
    u["Region"] = u["Location"].map(to_region)
    u["ETF_Ticker"] = u[c_etf].astype(str)
    u["company_ticker"] = u[c_tic].astype(str)
    u["company_name"] = (u[c_name] if c_name else u["company_ticker"]).astype(str)
    u["is_clean"] = (u.get("Clean", 0).astype(int) == 1).astype(int)
    u["is_contro"] = (u.get("Controversies", 0).astype(int) == 1).astype(int)

    # 3) covariance (optional)
    cov = None
    if os.path.exists(COV_CSV):
        cov = pd.read_csv(COV_CSV, index_col=0)
        cov = cov.replace([np.inf,-np.inf], np.nan).fillna(0.0)

    all_deltas = []
    all_metrics = []
    prog_rows = [{"ts": time.time(), "step": "write_specs_csv", "detail": SPECS_CSV}]

    # 4) scenarios
    for sc in SCENARIOS:
        sname    = sc["name"]
        cap_pct  = sc["single_name_cap_pct"]
        cap_mult = sc["single_name_cap_mult"]
        te_budget= sc["te_budget_annual"]
        sec_band = sc["sector_neutrality_pct"]
        reg_band = sc["region_neutrality_pct"]

        port_rows = []
        for etf, g0 in u.groupby("ETF_Ticker", sort=False):
            g = g0[["ETF_Ticker","company_ticker","company_name","Sector","Region","Location","w_base","is_clean","is_contro"]].copy()
            base_sector_sums = g.groupby("Sector", dropna=False)["w_base"].sum().to_dict()
            base_region_sums = g.groupby("Region", dropna=False)["w_base"].sum().to_dict()

            if sc["type"] == "baseline":
                g1 = scenario_apply_baseline(g)
            elif sc["type"] == "exclude":
                g1 = scenario_apply_exclude(g, sc["hard_screens"], cap_pct, cap_mult,
                                            base_sector_sums, base_region_sums, sec_band, reg_band)
            elif sc["type"] == "tilt":
                # TE-constrained bisection (best feasible alpha)
                lo, hi = 0.0, 1.0
                best = None
                for _ in range(16):
                    mid = (lo+hi)/2
                    g_try = scenario_apply_tilt(g, mid, cap_pct, cap_mult,
                                                base_sector_sums, base_region_sums, sec_band, reg_band)
                    if cov is not None:
                        b_vec = to_port_vec(g, "company_ticker", "w_base")
                        s_vec = to_port_vec(g_try, "company_ticker", "w_new")
                        te = tracking_error_annual(b_vec, s_vec, cov)
                    else:
                        te = np.nan

                    if te_budget is None or (not np.isnan(te) and te <= te_budget):
                        best = (mid, g_try.copy(), te)
                        lo = mid
                    else:
                        hi = mid
                g1 = (best[1] if best else scenario_apply_tilt(g, 0.0, cap_pct, cap_mult,
                                                               base_sector_sums, base_region_sums, sec_band, reg_band))
            else:
                g1 = g.copy(); g1["w_new"] = g1["w_base"]

            g1["scenario"] = sname
            port_rows.append(g1)

        port = pd.concat(port_rows, ignore_index=True)

        # Sanity: non-negatives & renorm per ETF
        port["w_new"] = port["w_new"].clip(lower=0.0)
        sums_base = port.groupby("ETF_Ticker", dropna=False)["w_base"].sum()
        sums_new  = port.groupby("ETF_Ticker", dropna=False)["w_new"].sum()
        fix_factors = (sums_base / sums_new.replace({0.0: np.nan})).fillna(1.0).to_dict()
        port["w_new"] = port["w_new"] * port["ETF_Ticker"].map(fix_factors)

        port["delta"] = port["w_new"] - port["w_base"]

        # 5) Metrics (per ETF)
        met_rows = []
        for etf, gx in port.groupby("ETF_Ticker", sort=False):
            sx = summarize_portfolio(gx)

            # TE & coverage
            if cov is not None:
                b_vec = to_port_vec(gx.assign(w_base=gx["w_base"]), "company_ticker", "w_base")
                s_vec = to_port_vec(gx.assign(w_new=gx["w_new"]), "company_ticker", "w_new")
                te = tracking_error_annual(b_vec, s_vec, cov)
                # coverage = share of baseline weight present in covariance universe
                covered = [t for t in b_vec.index if (t in cov.index and t in cov.columns)]
                te_coverage = float(b_vec.reindex(covered).fillna(0.0).sum())
            else:
                te = np.nan
                te_coverage = np.nan

            # Active metrics
            active_share = 0.5 * float(np.abs(gx["w_new"] - gx["w_base"]).sum())
            pct_active_names = 100.0 * float((gx["delta"].abs() > 0).sum()) / max(1.0, len(gx))
            max_delta_row = gx.iloc[(gx["delta"].abs().values).argmax()] if len(gx) else None
            max_delta_name = (max_delta_row["company_ticker"] if max_delta_row is not None else "")

            met = {
                "scenario": sname,
                "ETF_Ticker": etf,
                "TE_annual": te,
                "TE_to_Budget": (te / sc["te_budget_annual"] if (sc["te_budget_annual"] not in (None,0) and not np.isnan(te)) else np.nan),
                "TE_coverage_wt": te_coverage,
                "%Clean": sx["pct_clean"],
                "%Controversial": sx["pct_contro"],
                "%Other": sx["pct_other"],
                "#names": int((gx["w_new"]>0).sum()),
                "ActiveShare_%": active_share * 100.0,
                "%ActiveNames": pct_active_names,
                "MaxDeltaName": max_delta_name,
                "MaxDelta_%": float((max_delta_row["delta"]*100.0) if max_delta_row is not None else 0.0)
            }

            # Sector/Region deviations in %
            sec_base = gx.groupby("Sector", dropna=False)["w_base"].sum()
            sec_new  = gx.groupby("Sector", dropna=False)["w_new"].sum()
            sec_dev  = (sec_new - sec_base).reindex(sorted(set(sec_base.index)|set(sec_new.index))).fillna(0.0)
            for k,v in sec_dev.items():
                met[f"sector_dev__{k}"] = float(v*100.0)

            reg_base = gx.groupby("Region", dropna=False)["w_base"].sum()
            reg_new  = gx.groupby("Region", dropna=False)["w_new"].sum()
            reg_dev  = (reg_new - reg_base).reindex(sorted(set(reg_base.index)|set(reg_new.index))).fillna(0.0)
            for k,v in reg_dev.items():
                met[f"region_dev__{k}"] = float(v*100.0)

            met_rows.append(met)

        all_metrics.append(pd.DataFrame(met_rows))
        all_deltas.append(
            port[["scenario","ETF_Ticker","company_ticker","company_name","Sector","Region","Location","w_base","w_new","delta"]].copy()
        )
        prog_rows.append({"ts": time.time(), "step": "scenario_done", "detail": sname})

    # 6) Write outputs
    pd.concat(all_metrics, ignore_index=True).to_csv(METRICS_CSV, index=False)
    pd.concat(all_deltas,  ignore_index=True).to_csv(DELTAS_CSV,  index=False)
    pd.DataFrame(prog_rows).to_csv(PROG_CSV, index=False)

    print("Wrote:")
    print("-", SPECS_CSV)
    print("-", METRICS_CSV)
    print("-", DELTAS_CSV)
    print("-", PROG_CSV)

if __name__ == "__main__":
    main()


Wrote:
- /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 3/scenario_specs.csv
- /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 3/scenario_portfolio_metrics.csv
- /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 3/scenario_position_deltas.csv
- /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 3/scenario_progress.csv
